# Bibliotecas

In [ ]:
import sys
sys.path.append("../libs/")
sys.path.append("../")

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.colors as pc
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeClassifierCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import confusion_matrix, classification_report, adjusted_rand_score, normalized_mutual_info_score, make_scorer, f1_score
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, cross_validate
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors

from scipy.signal import find_peaks
from scipy.integrate import trapezoid
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist
from scipy.stats import skew, kurtosis, gaussian_kde
from scipy import stats

from hampel import hampel

from aeon.transformations.collection.convolution_based import Rocket

from tslearn.clustering import KShape
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from tslearn.utils import to_time_series_dataset

warnings.filterwarnings('ignore')

DIR_DATA = os.getcwd()+"/data/"
DIR_OUTPUT = os.getcwd()+"/output/"

# Carregando dados

## Crystallizer #1

In [ ]:
base_name_crystallizer1 = "Crystallizer #1.csv"

df_crystallizer1 = pd.read_csv(DIR_DATA + base_name_crystallizer1, sep=";", decimal=".")
df_crystallizer1["TIMESTAMP"] = pd.to_datetime(df_crystallizer1["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_crystallizer1["Resultado de Ferro (ppm)"] = pd.to_numeric(df_crystallizer1["Resultado de Ferro (ppm)"], errors="coerce")
df_crystallizer1.sort_values(by="TIMESTAMP", inplace=True)

# Verificação de amostras duplicadas
df_duplicados_crystallizer1 = df_crystallizer1[df_crystallizer1.duplicated(subset=['Labref'], keep=False)]

# Retirando linhas 23 e 2141 que estão duplicadas mas não possuem amostras significativas
df_crystallizer1.drop([23,2141], inplace=True) 
# Aplicando média para medidas com Labref iguais
agg_logic = {col: 'mean' if df_crystallizer1[col].dtype.kind in 'biufc' else 'first' 
             for col in df_crystallizer1.columns if col != 'Labref'}
df_crystallizer1 = df_crystallizer1.groupby('Labref', as_index=False).agg(agg_logic)

# Removendo amostras com valores muito discrepantes, 100 ppm definido como um limite
remove = df_crystallizer1[df_crystallizer1["Resultado de Ferro (ppm)"] > 100]
print(f"Amostras acima de 100 ppm: {len(remove)}")
remove[["Labref", "TIMESTAMP", "Resultado de Ferro (ppm)"]]
df_crystallizer1 = df_crystallizer1.drop(remove.index)
df_crystallizer1

## Crystallizer #2

In [ ]:
base_name_crystallizer2 = "Crystallizer #2.csv"

df_crystallizer2 = pd.read_csv(DIR_DATA + base_name_crystallizer2, sep=";", decimal=".")
df_crystallizer2["TIMESTAMP"] = pd.to_datetime(df_crystallizer2["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_crystallizer2["Resultado de Ferro (ppm)"] = pd.to_numeric(df_crystallizer2["Resultado de Ferro (ppm)"], errors="coerce")
df_crystallizer2.sort_values(by="TIMESTAMP", inplace=True)

# Verificação de amostras duplicadas
df_duplicados_crystallizer2 = df_crystallizer2[df_crystallizer2.duplicated(subset=['Labref'], keep=False)]

# Aplicando média para medidas com Labref iguais
agg_logic = {col: 'mean' if df_crystallizer2[col].dtype.kind in 'biufc' else 'first' 
             for col in df_crystallizer2.columns if col != 'Labref'}
df_crystallizer2 = df_crystallizer2.groupby('Labref', as_index=False).agg(agg_logic)

# Removendo amostra 4027521 que possui valor muito discrepante
idx = df_crystallizer2[df_crystallizer2['Labref'] == 4027521].index 
df_crystallizer2 = df_crystallizer2.drop(idx)


# Removendo amostras com valores muito discrepantes, 100 ppm definido como um limite
remove = df_crystallizer2[df_crystallizer2["Resultado de Ferro (ppm)"] > 100]
print(f"Amostras acima de 100 ppm: {len(remove)}")
remove[["Labref", "TIMESTAMP", "Resultado de Ferro (ppm)"]]
df_crystallizer2 = df_crystallizer2.drop(remove.index)
df_crystallizer2

## Crystallizer #3

In [ ]:
base_name_crystallizer3 = "Crystallizer #3.csv"

df_crystallizer3 = pd.read_csv(DIR_DATA + base_name_crystallizer3, sep=";", decimal=".")
df_crystallizer3["TIMESTAMP"] = pd.to_datetime(df_crystallizer3["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_crystallizer3["Resultado de Ferro (ppm)"] = pd.to_numeric(df_crystallizer3["Resultado de Ferro (ppm)"], errors="coerce")
df_crystallizer3.sort_values(by="TIMESTAMP", inplace=True)

# Verificação de amostras duplicadas
df_duplicados_crystallizer3 = df_crystallizer3[df_crystallizer3.duplicated(subset=['Labref'], keep=False)]


# Retirando linhas 6476 e 6494 que estão duplicadas mas não possuem amostras significativas
df_crystallizer3.drop([6476,6494], inplace=True) 
# Aplicando média para medidas com Labref iguais
agg_logic = {col: 'mean' if df_crystallizer3[col].dtype.kind in 'biufc' else 'first' 
             for col in df_crystallizer3.columns if col != 'Labref'}
df_crystallizer3 = df_crystallizer3.groupby('Labref', as_index=False).agg(agg_logic)

# Removendo amostras com valores muito discrepantes, 100 ppm definido como um limite
remove = df_crystallizer3[df_crystallizer3["Resultado de Ferro (ppm)"] > 100]
print(f"Amostras acima de 100 ppm: {len(remove)}")
remove[["Labref", "TIMESTAMP", "Resultado de Ferro (ppm)"]]
df_crystallizer3 = df_crystallizer3.drop(remove.index)
df_crystallizer3

# Carregando eventos identificados

## Deifinindo LC

In [ ]:
threshold = 5

## Crystallizer #1

In [ ]:
base_name_eventos_crystallizer1 = "Eventos-Reator1.csv"

df_eventos_crystallizer1 = pd.read_csv(DIR_DATA + base_name_eventos_crystallizer1, sep=";", decimal=".")
df_eventos_crystallizer1["TIMESTAMP"] = pd.to_datetime(df_eventos_crystallizer1["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_eventos_crystallizer1["Real"] = 1

nova_linha = {
    "TIMESTAMP": pd.to_datetime("2025-10-21 08:38:00"),
    "Real": 0,
    "Evento": "Falso Alarme"
}

df_eventos_crystallizer1 = pd.concat([df_eventos_crystallizer1, pd.DataFrame([nova_linha])], ignore_index=True)
df_eventos_crystallizer1 = df_eventos_crystallizer1.sort_values("TIMESTAMP").reset_index(drop=True)

# Removendo eventos que não são trocas de reator
df_eventos_crystallizer1.drop([2,3,4,6,7,8], inplace=True) 
df_eventos_crystallizer1

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

df_over = df_crystallizer1[df_crystallizer1["Resultado de Ferro (ppm)"] > threshold].copy()
df_over = df_over.sort_values("TIMESTAMP")

eventos_detectados = []
last_date = None

for _, row in df_over.iterrows():
    candidato = row["TIMESTAMP"]
    
    # Ignora se muito próximo do último evento detectado neste loop
    if last_date is not None and (candidato - last_date).days < DIAS_BASELINE:
        continue
    
    # Ignora se já existe um evento próximo no df_eventos_crystallizer1 original
    diffs = (df_eventos_crystallizer1["TIMESTAMP"] - candidato).abs()
    ja_existe = (diffs <= pd.Timedelta(days=DIAS_BASELINE)).any()
    if ja_existe:
        last_date = candidato  # avança o ponteiro para evitar acúmulo
        continue
    
    eventos_detectados.append(candidato)
    last_date = candidato

# Cria dataframe com os eventos detectados
df_novos_eventos = pd.DataFrame({
    "TIMESTAMP": eventos_detectados,
    "Evento": "Ultrapassagem Fe > 5ppm mas sem problema relatado",
    "Real": 0
})

# Adiciona ao df_eventos_crystallizer1 existente
df_eventos_crystallizer1 = pd.concat([df_eventos_crystallizer1, df_novos_eventos], ignore_index=True)
df_eventos_crystallizer1 = df_eventos_crystallizer1.sort_values("TIMESTAMP").reset_index(drop=True)
df_eventos_crystallizer1

## Crystallizer #2

In [ ]:
base_name_eventos_crystallizer2 = "Eventos-Reator2.csv"

df_eventos_crystallizer2 = pd.read_csv(DIR_DATA + base_name_eventos_crystallizer2, sep=";", decimal=".")
df_eventos_crystallizer2["TIMESTAMP"] = pd.to_datetime(df_eventos_crystallizer2["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_eventos_crystallizer2["Real"] = 1

# Removendo eventos fora do período de dados
df_eventos_crystallizer2.drop([0,1,2], inplace=True) 

# Removendo eventos que não são trocas de reator
df_eventos_crystallizer2.drop([6,9], inplace=True) 
df_eventos_crystallizer2

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

df_over = df_crystallizer2[df_crystallizer2["Resultado de Ferro (ppm)"] > threshold].copy()
df_over = df_over.sort_values("TIMESTAMP")

eventos_detectados = []
last_date = None

for _, row in df_over.iterrows():
    candidato = row["TIMESTAMP"]
    
    # Ignora se muito próximo do último evento detectado neste loop
    if last_date is not None and (candidato - last_date).days < DIAS_BASELINE:
        continue
    
    # Ignora se já existe um evento próximo no df_eventos_crystallizer2 original
    diffs = (df_eventos_crystallizer2["TIMESTAMP"] - candidato).abs()
    ja_existe = (diffs <= pd.Timedelta(days=DIAS_BASELINE)).any()
    if ja_existe:
        last_date = candidato  # avança o ponteiro para evitar acúmulo
        continue
    
    eventos_detectados.append(candidato)
    last_date = candidato

# Cria dataframe com os eventos detectados
df_novos_eventos = pd.DataFrame({
    "TIMESTAMP": eventos_detectados,
    "Evento": "Ultrapassagem Fe > 5ppm mas sem problema relatado",
    "Real": 0
})

# Adiciona ao df_eventos_crystallizer2 existente
df_eventos_crystallizer2 = pd.concat([df_eventos_crystallizer2, df_novos_eventos], ignore_index=True)
df_eventos_crystallizer2 = df_eventos_crystallizer2.sort_values("TIMESTAMP").reset_index(drop=True)
df_eventos_crystallizer2

## Crystallizer #3

In [ ]:
base_name_eventos_crystallizer3 = "Eventos-Reator3.csv"

df_eventos_crystallizer3 = pd.read_csv(DIR_DATA + base_name_eventos_crystallizer3, sep=";", decimal=".")
df_eventos_crystallizer3["TIMESTAMP"] = pd.to_datetime(df_eventos_crystallizer3["TIMESTAMP"],format="%Y-%m-%d %H:%M:%S")
df_eventos_crystallizer3["Real"] = 1

df_eventos_crystallizer3.drop([0,1], inplace=True) # Removendo eventos fora do período de dados

# Removendo eventos que não são trocas de reator
df_eventos_crystallizer3.drop([4,7], inplace=True) 
df_eventos_crystallizer3

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

df_over = df_crystallizer3[df_crystallizer3["Resultado de Ferro (ppm)"] > threshold].copy()
df_over = df_over.sort_values("TIMESTAMP")

eventos_detectados = []
last_date = None

for _, row in df_over.iterrows():
    candidato = row["TIMESTAMP"]
    
    # Ignora se muito próximo do último evento detectado neste loop
    if last_date is not None and (candidato - last_date).days < DIAS_BASELINE:
        continue
    
    # Ignora se já existe um evento próximo no df_eventos_crystallizer3 original
    diffs = (df_eventos_crystallizer3["TIMESTAMP"] - candidato).abs()
    ja_existe = (diffs <= pd.Timedelta(days=DIAS_BASELINE)).any()
    if ja_existe:
        last_date = candidato  # avança o ponteiro para evitar acúmulo
        continue
    
    eventos_detectados.append(candidato)
    last_date = candidato

# Cria dataframe com os eventos detectados
df_novos_eventos = pd.DataFrame({
    "TIMESTAMP": eventos_detectados,
    "Evento": "Ultrapassagem Fe > 5ppm mas sem problema relatado",
    "Real": 0
})

# Adiciona ao df_eventos_crystallizer3 existente
df_eventos_crystallizer3 = pd.concat([df_eventos_crystallizer3, df_novos_eventos], ignore_index=True)
df_eventos_crystallizer3 = df_eventos_crystallizer3.sort_values("TIMESTAMP").reset_index(drop=True)
df_eventos_crystallizer3

# Plotando Gráfico das medições
Linhas vermelhas = Eventos relatados  
Linhas azuis = Eventos de ultapassagem sem relatos

## Crystallizer #1

In [ ]:
fig_crystallizer1 = go.Figure()
fig_crystallizer1.add_trace(go.Scatter(
    x=df_crystallizer1['TIMESTAMP'],
    y=df_crystallizer1["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer1.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer1.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer1.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer1.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #1",
)
fig_crystallizer1.show()

In [ ]:
# fig_crystallizer1.write_html("Crystallizer #1.html")

## Crystallizer #2

In [ ]:
fig_crystallizer2 = go.Figure()
fig_crystallizer2.add_trace(go.Scatter(
    x=df_crystallizer2['TIMESTAMP'],
    y=df_crystallizer2["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer2.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer2.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer2.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #2",
)
fig_crystallizer2.show()

In [ ]:
# fig_crystallizer2.write_html("Crystallizer #2.html")

## Crystallizer #3

In [ ]:
fig_crystallizer3 = go.Figure()
fig_crystallizer3.add_trace(go.Scatter(
    x=df_crystallizer3['TIMESTAMP'],
    y=df_crystallizer3["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig_crystallizer3.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_crystallizer3.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_crystallizer3.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top",
        font=dict(color=cor)  # anotação com a mesma cor da linha
    )

fig_crystallizer3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizer #3",
)
fig_crystallizer3.show()

In [ ]:
# fig_crystallizer3.write_html("Crystallizer #3.html")

## Crystallizer #1#2#3

In [ ]:
crystallizers = [
    {"df": df_crystallizer1, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1", "cor": "black"},
    {"df": df_crystallizer2, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2", "cor": "steelblue"},
    {"df": df_crystallizer3, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3", "cor": "green"},
]

fig = go.Figure()

for c in crystallizers:
    # Série principal
    fig.add_trace(go.Scatter(
        x=c["df"]['TIMESTAMP'],
        y=c["df"]["Resultado de Ferro (ppm)"],
        mode='lines',
        name=c["nome"],
        line=dict(color=c["cor"])
    ))

    # Linhas verticais dos eventos
    for _, row in c["df_eventos"].iterrows():
        cor = c["cor"] if row["Real"] == 1 else "gray"
        fig.add_shape(
            type="line",
            x0=str(row["TIMESTAMP"]),
            x1=str(row["TIMESTAMP"]),
            y0=0,
            y1=1,
            yref="paper",
            line=dict(color=cor, width=1.5, dash="dash")
        )
        fig.add_annotation(
            x=str(row["TIMESTAMP"]),
            y=1,
            yref="paper",
            text=row["EVENTO"] if "EVENTO" in c["df_eventos"].columns else "",
            showarrow=False,
            textangle=-90,
            yanchor="top",
            font=dict(color=cor)
        )

fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) - Crystallizers #1, #2 e #3"
)
fig.show()

In [ ]:
# fig.write_html("Crystallizer #1#2#3.html")

## Gráfico com Média Móvel
Verificando se há tendência clara nos dados

## MM Crystallizer #1

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer1 = df_crystallizer1.copy()
df_mm_crystallizer1 = df_mm_crystallizer1.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer1[f'MM_{dias}D'] = df_mm_crystallizer1["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer1 = df_mm_crystallizer1.reset_index()

fig_mm_crystallizer1 = go.Figure()

# Série original
fig_mm_crystallizer1.add_trace(go.Scatter(
    x=df_mm_crystallizer1['TIMESTAMP'],
    y=df_mm_crystallizer1["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer1.add_trace(go.Scatter(
        x=df_mm_crystallizer1['TIMESTAMP'],
        y=df_mm_crystallizer1[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer1.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer1.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer1.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer1.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer1.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer1.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #1"
)
fig_mm_crystallizer1.show()

In [ ]:
# fig_mm_crystallizer1.write_html("Crystallizer #1_MM.html")

## MM Crystallizer #2

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer2 = df_crystallizer2.copy()
df_mm_crystallizer2 = df_mm_crystallizer2.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer2[f'MM_{dias}D'] = df_mm_crystallizer2["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer2 = df_mm_crystallizer2.reset_index()

fig_mm_crystallizer2 = go.Figure()

# Série original
fig_mm_crystallizer2.add_trace(go.Scatter(
    x=df_mm_crystallizer2['TIMESTAMP'],
    y=df_mm_crystallizer2["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer2.add_trace(go.Scatter(
        x=df_mm_crystallizer2['TIMESTAMP'],
        y=df_mm_crystallizer2[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer2.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer2.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer2.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer2.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer2.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer2.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #2"
)
fig_mm_crystallizer2.show()

In [ ]:
# fig_mm_crystallizer2.write_html("Crystallizer #2_MM.html")

## MM Crystallizer #3

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
CORES_MM = ['blue', 'green', 'orange', 'purple', 'red']

df_mm_crystallizer3 = df_crystallizer3.copy()
df_mm_crystallizer3 = df_mm_crystallizer3.set_index('TIMESTAMP')

for dias in JANELAS:
    df_mm_crystallizer3[f'MM_{dias}D'] = df_mm_crystallizer3["Resultado de Ferro (ppm)"].rolling(window=f'{dias}D').mean()

df_mm_crystallizer3 = df_mm_crystallizer3.reset_index()

fig_mm_crystallizer3 = go.Figure()

# Série original
fig_mm_crystallizer3.add_trace(go.Scatter(
    x=df_mm_crystallizer3['TIMESTAMP'],
    y=df_mm_crystallizer3["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='gray', width=1),
    opacity=0.6
))

# Médias móveis
for dias, cor in zip(JANELAS, CORES_MM):
    fig_mm_crystallizer3.add_trace(go.Scatter(
        x=df_mm_crystallizer3['TIMESTAMP'],
        y=df_mm_crystallizer3[f'MM_{dias}D'],
        mode='lines',
        name=f"MM {dias}D",
        line=dict(color=cor, width=2)
    ))

fig_mm_crystallizer3.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos_crystallizer3.iterrows():
    cor = "red" if row["Real"] == 1 else "blue"
    fig_mm_crystallizer3.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )
    fig_mm_crystallizer3.add_annotation(
        x=str(row["TIMESTAMP"]),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos_crystallizer3.columns else "",
        showarrow=False,
        
        textangle=-90,
        yanchor="top"
    )

fig_mm_crystallizer3.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title="Fe (ppm) MM - Crystallizer #3"
)
fig_mm_crystallizer3.show()

In [ ]:
# fig_mm_crystallizer3.write_html("Crystallizer #3_MM.html")

## MM Crystallizer #1#2#3

In [ ]:
NUM_DIAS = 7  # parâmetro da janela

CORES = {
    "Crystallizer #1": "blue",
    "Crystallizer #2": "green",
    "Crystallizer #3": "orange"
}

crystallizers = [
    {"df": df_crystallizer1, "df_eventos": df_eventos_crystallizer1, "nome": "Crystallizer #1"},
    {"df": df_crystallizer2, "df_eventos": df_eventos_crystallizer2, "nome": "Crystallizer #2"},
    {"df": df_crystallizer3, "df_eventos": df_eventos_crystallizer3, "nome": "Crystallizer #3"},
]

fig_mm_123 = go.Figure()

for c in crystallizers:
    cor = CORES[c["nome"]]

    # Calcula média móvel
    df_mm = c["df"].copy().set_index('TIMESTAMP')
    df_mm[f'MM_{NUM_DIAS}D'] = df_mm["Resultado de Ferro (ppm)"].rolling(window=f'{NUM_DIAS}D').mean()
    df_mm = df_mm.reset_index()

    # Série original
    fig_mm_123.add_trace(go.Scatter(
        x=df_mm['TIMESTAMP'],
        y=df_mm["Resultado de Ferro (ppm)"],
        mode='lines',
        name=f"{c['nome']} — original",
        line=dict(color=cor, width=1),
        opacity=0.3
    ))

    # Média móvel
    fig_mm_123.add_trace(go.Scatter(
        x=df_mm['TIMESTAMP'],
        y=df_mm[f'MM_{NUM_DIAS}D'],
        mode='lines',
        name=f"{c['nome']} — MM {NUM_DIAS}D",
        line=dict(color=cor, width=2)
    ))

    # Linhas verticais dos eventos
    for _, row in c["df_eventos"].iterrows():
        dash_evento = "solid" if row["Real"] == 1 else "dash"
        fig_mm_123.add_shape(
            type="line",
            x0=str(row["TIMESTAMP"]),
            x1=str(row["TIMESTAMP"]),
            y0=0, y1=1,
            yref="paper",
            line=dict(color=cor, width=1.5, dash=dash_evento)
        )
        fig_mm_123.add_annotation(
            x=str(row["TIMESTAMP"]),
            y=1,
            yref="paper",
            text=row["EVENTO"] if "EVENTO" in c["df_eventos"].columns else "",
            showarrow=False,
            textangle=-90,
            yanchor="top",
            font=dict(color=cor)
        )

fig_mm_123.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

fig_mm_123.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title=f"Fe (ppm) MM {NUM_DIAS}D — Crystallizers #1, #2 e #3"
)
fig_mm_123.show()

In [ ]:
# fig_mm_123.write_html("Crystallizer_7DMM_#1#2#3.html")

# Estatísticas descritivas de toda série de concentração de Fe

## Crystallizer #1

In [ ]:
serie_crystallizer1 = df_crystallizer1["Resultado de Ferro (ppm)"].dropna()

descritivas_crystallizer1 = pd.DataFrame({
    "Métrica": [
        "Contagem", "Média", "Mediana", "Desvio Padrão", "Variância",
        "Mínimo", "Máximo", "Amplitude",
        "Q1 (25%)", "Q3 (75%)", "IQR",
        "Assimetria", "Curtose"
    ],
    "Valor": [
        serie_crystallizer1.count(), serie_crystallizer1.mean(), serie_crystallizer1.median(), serie_crystallizer1.std(), serie_crystallizer1.var(),
        serie_crystallizer1.min(), serie_crystallizer1.max(), serie_crystallizer1.max() - serie_crystallizer1.min(),
        serie_crystallizer1.quantile(0.25), serie_crystallizer1.quantile(0.75), serie_crystallizer1.quantile(0.75) - serie_crystallizer1.quantile(0.25),
        serie_crystallizer1.skew(), serie_crystallizer1.kurt()
    ]
})

descritivas_crystallizer1["Valor"] = descritivas_crystallizer1["Valor"].round(4)
descritivas_crystallizer1.set_index("Métrica", inplace=True)
descritivas_crystallizer1.T

### Histograma

In [ ]:
fig_hist_crystallizer1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Histograma + KDE", "Violin Plot"]
)

# Histograma
fig_hist_crystallizer1.add_trace(go.Histogram(
    x=serie_crystallizer1,
    nbinsx=670,
    name="Histograma",
    marker_color='steelblue',
    opacity=0.7
), row=1, col=1)

# KDE sobre o histograma
kde = gaussian_kde(serie_crystallizer1)
x_range = np.linspace(serie_crystallizer1.min(), serie_crystallizer1.max(), 1000)
y_kde = kde(x_range)
# Escala a KDE para ficar na mesma escala do histograma
y_kde_scaled = y_kde * len(serie_crystallizer1) * (serie_crystallizer1.max() - serie_crystallizer1.min()) / 670

fig_hist_crystallizer1.add_trace(go.Scatter(
    x=x_range,
    y=y_kde_scaled,
    mode='lines',
    line=dict(color='darkblue', width=2),
    name="KDE"
), row=1, col=1)

# Violin Plot
fig_hist_crystallizer1.add_trace(go.Violin(
    y=serie_crystallizer1,
    name="Violin",
    marker_color='steelblue',
    box_visible=True,    # boxplot interno
    meanline_visible=True
), row=1, col=2)

fig_hist_crystallizer1.update_layout(
    height=600,
    template='plotly_white',
    title="Análise Descritiva — Resultado de Ferro (ppm) Cristallyzer #1",
    showlegend=False
)
fig_hist_crystallizer1.show()

### Teste de normalidade

In [ ]:
# Visualização
fig_normalidade_crystallizer1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Q-Q Plot", "Histograma + Distribuição Normal Teórica"]
)

# Q-Q Plot
qq = stats.probplot(serie_crystallizer1, dist="norm")
qq_x = [qq[0][0][0], qq[0][0][-1]]
qq_y = [qq[1][1] + qq[1][0] * qq[0][0][0],
        qq[1][1] + qq[1][0] * qq[0][0][-1]]

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=qq[0][0], y=qq[0][1],
    mode='markers',
    marker=dict(color='steelblue', size=4, opacity=0.5),
    name='Quantis observados'
), row=1, col=1)

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=qq_x, y=qq_y,
    mode='lines',
    line=dict(color='red', width=2),
    name='Linha normal teórica'
), row=1, col=1)

# Histograma + curva normal teórica
x_range = np.linspace(serie_crystallizer1.min(), serie_crystallizer1.max(), 300)
y_normal = stats.norm.pdf(x_range, serie_crystallizer1.mean(), serie_crystallizer1.std())
y_normal_scaled = y_normal * len(serie_crystallizer1) * (serie_crystallizer1.max() - serie_crystallizer1.min()) / 50

fig_normalidade_crystallizer1.add_trace(go.Histogram(
    x=serie_crystallizer1, nbinsx=50,
    marker_color='steelblue', opacity=0.6,
    name='Dados observados'
), row=1, col=2)

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=x_range, y=y_normal_scaled,
    mode='lines',
    line=dict(color='red', width=2),
    name='Normal teórica'
), row=1, col=2)

fig_normalidade_crystallizer1.update_xaxes(title_text="Quantis teóricos", row=1, col=1)
fig_normalidade_crystallizer1.update_yaxes(title_text="Quantis observados", row=1, col=1)
fig_normalidade_crystallizer1.update_xaxes(title_text="Resultado de Ferro (ppm)", row=1, col=2)
fig_normalidade_crystallizer1.update_yaxes(title_text="Contagem", row=1, col=2)

fig_normalidade_crystallizer1.update_layout(
    height=500,
    template='plotly_white',
    title="Análise de Normalidade — Resultado de Ferro (ppm) Cristallyzer #1"
)
fig_normalidade_crystallizer1.show()

### Análise sem outliers (Método IQR)

In [ ]:
# Remove outliers via IQR
Q1 = serie_crystallizer1.quantile(0.25)
Q3 = serie_crystallizer1.quantile(0.75)
IQR = Q3 - Q1
serie_iqr_crystallizer1 = serie_crystallizer1[(serie_crystallizer1 >= Q1 - 1.5 * IQR) & (serie_crystallizer1 <= Q3 + 1.5 * IQR)]

print(f"Amostras originais:    {len(serie_crystallizer1)}")
print(f"Amostras sem outliers: {len(serie_iqr_crystallizer1)} ({len(serie_crystallizer1) - len(serie_iqr_crystallizer1)} removidas)")

In [ ]:
descritivas = pd.DataFrame({
    "Métrica": [
        "Contagem", "Média", "Mediana", "Desvio Padrão", "Variância",
        "Mínimo", "Máximo", "Amplitude",
        "Q1 (25%)", "Q3 (75%)", "IQR",
        "Assimetria", "Curtose"
    ],
    "Valor": [
        serie_iqr_crystallizer1.count(), serie_iqr_crystallizer1.mean(), serie_iqr_crystallizer1.median(), serie_iqr_crystallizer1.std(), serie_iqr_crystallizer1.var(),
        serie_iqr_crystallizer1.min(), serie_iqr_crystallizer1.max(), serie_iqr_crystallizer1.max() - serie_iqr_crystallizer1.min(),
        serie_iqr_crystallizer1.quantile(0.25), serie_iqr_crystallizer1.quantile(0.75), serie_iqr_crystallizer1.quantile(0.75) - serie_iqr_crystallizer1.quantile(0.25),
        serie_iqr_crystallizer1.skew(), serie_iqr_crystallizer1.kurt()
    ]
})

descritivas["Valor"] = descritivas["Valor"].round(4)
descritivas.set_index("Métrica", inplace=True)
descritivas.T

#### Histograma sem outliers

In [ ]:
fig_iqr_crystallizer1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Histograma + KDE", "Violin Plot"]
)

fig_iqr_crystallizer1.add_trace(go.Histogram(
    x=serie_iqr_crystallizer1,
    nbinsx=70,
    name="Histograma",
    marker_color='steelblue',
    opacity=0.7
), row=1, col=1)

kde = gaussian_kde(serie_iqr_crystallizer1)
x_range = np.linspace(serie_iqr_crystallizer1.min(), serie_iqr_crystallizer1.max(), 1000)
y_kde = kde(x_range)
y_kde_scaled = y_kde * len(serie_iqr_crystallizer1) * (serie_iqr_crystallizer1.max() - serie_iqr_crystallizer1.min()) / 70

fig_iqr_crystallizer1.add_trace(go.Scatter(
    x=x_range,
    y=y_kde_scaled,
    mode='lines',
    line=dict(color='darkblue', width=2),
    name="KDE"
), row=1, col=1)

fig_iqr_crystallizer1.add_trace(go.Violin(
    y=serie_iqr_crystallizer1,
    name="Violin",
    marker_color='steelblue',
    box_visible=True,
    meanline_visible=True
), row=1, col=2)

fig_iqr_crystallizer1.update_layout(
    height=600,
    template='plotly_white',
    title="Análise Descritiva sem Outliers — Resultado de Ferro (ppm) Cristallyzer #1",
    showlegend=False
)
fig_iqr_crystallizer1.show()

#### Teste de normalidade sem outliers

In [ ]:
# Visualização
fig_iqr_normalidade_crystallizer1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Q-Q Plot", "Histograma + Distribuição Normal Teórica"]
)

# Q-Q Plot
qq = stats.probplot(serie_iqr_crystallizer1, dist="norm")
qq_x = [qq[0][0][0], qq[0][0][-1]]
qq_y = [qq[1][1] + qq[1][0] * qq[0][0][0],
        qq[1][1] + qq[1][0] * qq[0][0][-1]]

fig_iqr_normalidade_crystallizer1.add_trace(go.Scatter(
    x=qq[0][0], y=qq[0][1],
    mode='markers',
    marker=dict(color='steelblue', size=4, opacity=0.5),
    name='Quantis observados'
), row=1, col=1)

fig_iqr_normalidade_crystallizer1.add_trace(go.Scatter(
    x=qq_x, y=qq_y,
    mode='lines',
    line=dict(color='red', width=2),
    name='Linha normal teórica'
), row=1, col=1)

# Histograma + curva normal teórica
x_range = np.linspace(serie_iqr_crystallizer1.min(), serie_iqr_crystallizer1.max(), 300)
y_normal = stats.norm.pdf(x_range, serie_iqr_crystallizer1.mean(), serie_iqr_crystallizer1.std())
y_normal_scaled = y_normal * len(serie_iqr_crystallizer1) * (serie_iqr_crystallizer1.max() - serie_iqr_crystallizer1.min()) / 50

fig_iqr_normalidade_crystallizer1.add_trace(go.Histogram(
    x=serie_iqr_crystallizer1, nbinsx=50,
    marker_color='steelblue', opacity=0.6,
    name='Dados observados'
), row=1, col=2)

fig_iqr_normalidade_crystallizer1.add_trace(go.Scatter(
    x=x_range, y=y_normal_scaled,
    mode='lines',
    line=dict(color='red', width=2),
    name='Normal teórica'
), row=1, col=2)

fig_iqr_normalidade_crystallizer1.update_xaxes(title_text="Quantis teóricos", row=1, col=1)
fig_iqr_normalidade_crystallizer1.update_yaxes(title_text="Quantis observados", row=1, col=1)
fig_iqr_normalidade_crystallizer1.update_xaxes(title_text="Resultado de Ferro (ppm)", row=1, col=2)
fig_iqr_normalidade_crystallizer1.update_yaxes(title_text="Contagem", row=1, col=2)

fig_iqr_normalidade_crystallizer1.update_layout(
    height=500,
    template='plotly_white',
    title="Análise de Normalidade sem outliers — Resultado de Ferro (ppm) Cristallyzer #1"
)
fig_iqr_normalidade_crystallizer1.show()

## Crystallizer #2

In [ ]:
serie_crystallizer2 = df_crystallizer2["Resultado de Ferro (ppm)"].dropna()

descritivas_crystallizer2 = pd.DataFrame({
    "Métrica": [
        "Contagem", "Média", "Mediana", "Desvio Padrão", "Variância",
        "Mínimo", "Máximo", "Amplitude",
        "Q1 (25%)", "Q3 (75%)", "IQR",
        "Assimetria", "Curtose"
    ],
    "Valor": [
        serie_crystallizer2.count(), serie_crystallizer2.mean(), serie_crystallizer2.median(), serie_crystallizer2.std(), serie_crystallizer2.var(),
        serie_crystallizer2.min(), serie_crystallizer2.max(), serie_crystallizer2.max() - serie_crystallizer2.min(),
        serie_crystallizer2.quantile(0.25), serie_crystallizer2.quantile(0.75), serie_crystallizer2.quantile(0.75) - serie_crystallizer2.quantile(0.25),
        serie_crystallizer2.skew(), serie_crystallizer2.kurt()
    ]
})

descritivas_crystallizer2["Valor"] = descritivas_crystallizer2["Valor"].round(4)
descritivas_crystallizer2.set_index("Métrica", inplace=True)
descritivas_crystallizer2.T

### Histograma

In [ ]:
fig_hist_crystallizer2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Histograma + KDE", "Violin Plot"]
)

# Histograma
fig_hist_crystallizer2.add_trace(go.Histogram(
    x=serie_crystallizer2,
    nbinsx=670,
    name="Histograma",
    marker_color='steelblue',
    opacity=0.7
), row=1, col=1)

# KDE sobre o histograma
kde = gaussian_kde(serie_crystallizer2)
x_range = np.linspace(serie_crystallizer2.min(), serie_crystallizer2.max(), 1000)
y_kde = kde(x_range)
# Escala a KDE para ficar na mesma escala do histograma
y_kde_scaled = y_kde * len(serie_crystallizer2) * (serie_crystallizer2.max() - serie_crystallizer2.min()) / 670

fig_hist_crystallizer2.add_trace(go.Scatter(
    x=x_range,
    y=y_kde_scaled,
    mode='lines',
    line=dict(color='darkblue', width=2),
    name="KDE"
), row=1, col=1)

# Violin Plot
fig_hist_crystallizer2.add_trace(go.Violin(
    y=serie_crystallizer2,
    name="Violin",
    marker_color='steelblue',
    box_visible=True,    # boxplot interno
    meanline_visible=True
), row=1, col=2)

fig_hist_crystallizer2.update_layout(
    height=600,
    template='plotly_white',
    title="Análise Descritiva — Resultado de Ferro (ppm) Cristallyzer #2",
    showlegend=False
)
fig_hist_crystallizer2.show()

### Teste de normalidade

In [ ]:
# Visualização
fig_normalidade_crystallizer2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Q-Q Plot", "Histograma + Distribuição Normal Teórica"]
)

# Q-Q Plot
qq = stats.probplot(serie_crystallizer2, dist="norm")
qq_x = [qq[0][0][0], qq[0][0][-1]]
qq_y = [qq[1][1] + qq[1][0] * qq[0][0][0],
        qq[1][1] + qq[1][0] * qq[0][0][-1]]

fig_normalidade_crystallizer2.add_trace(go.Scatter(
    x=qq[0][0], y=qq[0][1],
    mode='markers',
    marker=dict(color='steelblue', size=4, opacity=0.5),
    name='Quantis observados'
), row=1, col=1)

fig_normalidade_crystallizer2.add_trace(go.Scatter(
    x=qq_x, y=qq_y,
    mode='lines',
    line=dict(color='red', width=2),
    name='Linha normal teórica'
), row=1, col=1)

# Histograma + curva normal teórica
x_range = np.linspace(serie_crystallizer2.min(), serie_crystallizer2.max(), 300)
y_normal = stats.norm.pdf(x_range, serie_crystallizer2.mean(), serie_crystallizer2.std())
y_normal_scaled = y_normal * len(serie_crystallizer2) * (serie_crystallizer2.max() - serie_crystallizer2.min()) / 50

fig_normalidade_crystallizer2.add_trace(go.Histogram(
    x=serie_crystallizer2, nbinsx=50,
    marker_color='steelblue', opacity=0.6,
    name='Dados observados'
), row=1, col=2)

fig_normalidade_crystallizer2.add_trace(go.Scatter(
    x=x_range, y=y_normal_scaled,
    mode='lines',
    line=dict(color='red', width=2),
    name='Normal teórica'
), row=1, col=2)

fig_normalidade_crystallizer2.update_xaxes(title_text="Quantis teóricos", row=1, col=1)
fig_normalidade_crystallizer2.update_yaxes(title_text="Quantis observados", row=1, col=1)
fig_normalidade_crystallizer2.update_xaxes(title_text="Resultado de Ferro (ppm)", row=1, col=2)
fig_normalidade_crystallizer2.update_yaxes(title_text="Contagem", row=1, col=2)

fig_normalidade_crystallizer2.update_layout(
    height=500,
    template='plotly_white',
    title="Análise de Normalidade — Resultado de Ferro (ppm) Cristallyzer #2"
)
fig_normalidade_crystallizer2.show()

### Análise sem outliers (Método IQR)

In [ ]:
# Remove outliers via IQR
Q1 = serie_crystallizer2.quantile(0.25)
Q3 = serie_crystallizer2.quantile(0.75)
IQR = Q3 - Q1
serie_iqr_crystallizer2 = serie_crystallizer2[(serie_crystallizer2 >= Q1 - 1.5 * IQR) & (serie_crystallizer2 <= Q3 + 1.5 * IQR)]

print(f"Amostras originais:    {len(serie_crystallizer2)}")
print(f"Amostras sem outliers: {len(serie_iqr_crystallizer2)} ({len(serie_crystallizer2) - len(serie_iqr_crystallizer2)} removidas)")

In [ ]:
descritivas = pd.DataFrame({
    "Métrica": [
        "Contagem", "Média", "Mediana", "Desvio Padrão", "Variância",
        "Mínimo", "Máximo", "Amplitude",
        "Q1 (25%)", "Q3 (75%)", "IQR",
        "Assimetria", "Curtose"
    ],
    "Valor": [
        serie_iqr_crystallizer2.count(), serie_iqr_crystallizer2.mean(), serie_iqr_crystallizer2.median(), serie_iqr_crystallizer2.std(), serie_iqr_crystallizer2.var(),
        serie_iqr_crystallizer2.min(), serie_iqr_crystallizer2.max(), serie_iqr_crystallizer2.max() - serie_iqr_crystallizer2.min(),
        serie_iqr_crystallizer2.quantile(0.25), serie_iqr_crystallizer2.quantile(0.75), serie_iqr_crystallizer2.quantile(0.75) - serie_iqr_crystallizer2.quantile(0.25),
        serie_iqr_crystallizer2.skew(), serie_iqr_crystallizer2.kurt()
    ]
})

descritivas["Valor"] = descritivas["Valor"].round(4)
descritivas.set_index("Métrica", inplace=True)
descritivas.T

#### Histograma sem outliers

In [ ]:
fig_iqr_crystallizer2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Histograma + KDE", "Violin Plot"]
)

fig_iqr_crystallizer2.add_trace(go.Histogram(
    x=serie_iqr_crystallizer2,
    nbinsx=62,
    name="Histograma",
    marker_color='steelblue',
    opacity=0.7
), row=1, col=1)

kde = gaussian_kde(serie_iqr_crystallizer2)
x_range = np.linspace(serie_iqr_crystallizer2.min(), serie_iqr_crystallizer2.max(), 1000)
y_kde = kde(x_range)
y_kde_scaled = y_kde * len(serie_iqr_crystallizer2) * (serie_iqr_crystallizer2.max() - serie_iqr_crystallizer2.min()) / 70

fig_iqr_crystallizer2.add_trace(go.Scatter(
    x=x_range,
    y=y_kde_scaled,
    mode='lines',
    line=dict(color='darkblue', width=2),
    name="KDE"
), row=1, col=1)

fig_iqr_crystallizer2.add_trace(go.Violin(
    y=serie_iqr_crystallizer2,
    name="Violin",
    marker_color='steelblue',
    box_visible=True,
    meanline_visible=True
), row=1, col=2)

fig_iqr_crystallizer2.update_layout(
    height=600,
    template='plotly_white',
    title="Análise Descritiva sem Outliers — Resultado de Ferro (ppm) Cristallyzer #2",
    showlegend=False
)
fig_iqr_crystallizer2.show()

#### Teste de normalidade sem outliers

In [ ]:
# Visualização
fig_iqr_normalidade_crystallizer2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Q-Q Plot", "Histograma + Distribuição Normal Teórica"]
)

# Q-Q Plot
qq = stats.probplot(serie_iqr_crystallizer2, dist="norm")
qq_x = [qq[0][0][0], qq[0][0][-1]]
qq_y = [qq[1][1] + qq[1][0] * qq[0][0][0],
        qq[1][1] + qq[1][0] * qq[0][0][-1]]

fig_iqr_normalidade_crystallizer2.add_trace(go.Scatter(
    x=qq[0][0], y=qq[0][1],
    mode='markers',
    marker=dict(color='steelblue', size=4, opacity=0.5),
    name='Quantis observados'
), row=1, col=1)

fig_iqr_normalidade_crystallizer2.add_trace(go.Scatter(
    x=qq_x, y=qq_y,
    mode='lines',
    line=dict(color='red', width=2),
    name='Linha normal teórica'
), row=1, col=1)

# Histograma + curva normal teórica
x_range = np.linspace(serie_iqr_crystallizer2.min(), serie_iqr_crystallizer2.max(), 300)
y_normal = stats.norm.pdf(x_range, serie_iqr_crystallizer2.mean(), serie_iqr_crystallizer2.std())
y_normal_scaled = y_normal * len(serie_iqr_crystallizer2) * (serie_iqr_crystallizer2.max() - serie_iqr_crystallizer2.min()) / 50

fig_iqr_normalidade_crystallizer2.add_trace(go.Histogram(
    x=serie_iqr_crystallizer2, nbinsx=50,
    marker_color='steelblue', opacity=0.6,
    name='Dados observados'
), row=1, col=2)

fig_iqr_normalidade_crystallizer2.add_trace(go.Scatter(
    x=x_range, y=y_normal_scaled,
    mode='lines',
    line=dict(color='red', width=2),
    name='Normal teórica'
), row=1, col=2)

fig_iqr_normalidade_crystallizer2.update_xaxes(title_text="Quantis teóricos", row=1, col=1)
fig_iqr_normalidade_crystallizer2.update_yaxes(title_text="Quantis observados", row=1, col=1)
fig_iqr_normalidade_crystallizer2.update_xaxes(title_text="Resultado de Ferro (ppm)", row=1, col=2)
fig_iqr_normalidade_crystallizer2.update_yaxes(title_text="Contagem", row=1, col=2)

fig_iqr_normalidade_crystallizer2.update_layout(
    height=500,
    template='plotly_white',
    title="Análise de Normalidade sem outliers — Resultado de Ferro (ppm) Cristallyzer #2"
)
fig_iqr_normalidade_crystallizer2.show()

## Crystallizer #3

In [ ]:
serie_crystallizer3 = df_crystallizer3["Resultado de Ferro (ppm)"].dropna()

descritivas_crystallizer3 = pd.DataFrame({
    "Métrica": [
        "Contagem", "Média", "Mediana", "Desvio Padrão", "Variância",
        "Mínimo", "Máximo", "Amplitude",
        "Q1 (25%)", "Q3 (75%)", "IQR",
        "Assimetria", "Curtose"
    ],
    "Valor": [
        serie_crystallizer3.count(), serie_crystallizer3.mean(), serie_crystallizer3.median(), serie_crystallizer3.std(), serie_crystallizer3.var(),
        serie_crystallizer3.min(), serie_crystallizer3.max(), serie_crystallizer3.max() - serie_crystallizer3.min(),
        serie_crystallizer3.quantile(0.25), serie_crystallizer3.quantile(0.75), serie_crystallizer3.quantile(0.75) - serie_crystallizer3.quantile(0.25),
        serie_crystallizer3.skew(), serie_crystallizer3.kurt()
    ]
})

descritivas_crystallizer3["Valor"] = descritivas_crystallizer3["Valor"].round(4)
descritivas_crystallizer3.set_index("Métrica", inplace=True)
descritivas_crystallizer3.T

### Histograma

In [ ]:
fig_hist_crystallizer3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Histograma + KDE", "Violin Plot"]
)

# Histograma
fig_hist_crystallizer3.add_trace(go.Histogram(
    x=serie_crystallizer3,
    nbinsx=670,
    name="Histograma",
    marker_color='steelblue',
    opacity=0.7
), row=1, col=1)

# KDE sobre o histograma
kde = gaussian_kde(serie_crystallizer3)
x_range = np.linspace(serie_crystallizer3.min(), serie_crystallizer3.max(), 1000)
y_kde = kde(x_range)
# Escala a KDE para ficar na mesma escala do histograma
y_kde_scaled = y_kde * len(serie_crystallizer3) * (serie_crystallizer3.max() - serie_crystallizer3.min()) / 670

fig_hist_crystallizer3.add_trace(go.Scatter(
    x=x_range,
    y=y_kde_scaled,
    mode='lines',
    line=dict(color='darkblue', width=2),
    name="KDE"
), row=1, col=1)

# Violin Plot
fig_hist_crystallizer3.add_trace(go.Violin(
    y=serie_crystallizer3,
    name="Violin",
    marker_color='steelblue',
    box_visible=True,    # boxplot interno
    meanline_visible=True
), row=1, col=2)

fig_hist_crystallizer3.update_layout(
    height=600,
    template='plotly_white',
    title="Análise Descritiva — Resultado de Ferro (ppm) Cristallyzer #3",
    showlegend=False
)
fig_hist_crystallizer3.show()

### Teste de normalidade

In [ ]:
# Visualização
fig_normalidade_crystallizer3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Q-Q Plot", "Histograma + Distribuição Normal Teórica"]
)

# Q-Q Plot
qq = stats.probplot(serie_crystallizer3, dist="norm")
qq_x = [qq[0][0][0], qq[0][0][-1]]
qq_y = [qq[1][1] + qq[1][0] * qq[0][0][0],
        qq[1][1] + qq[1][0] * qq[0][0][-1]]

fig_normalidade_crystallizer3.add_trace(go.Scatter(
    x=qq[0][0], y=qq[0][1],
    mode='markers',
    marker=dict(color='steelblue', size=4, opacity=0.5),
    name='Quantis observados'
), row=1, col=1)

fig_normalidade_crystallizer3.add_trace(go.Scatter(
    x=qq_x, y=qq_y,
    mode='lines',
    line=dict(color='red', width=2),
    name='Linha normal teórica'
), row=1, col=1)

# Histograma + curva normal teórica
x_range = np.linspace(serie_crystallizer3.min(), serie_crystallizer3.max(), 300)
y_normal = stats.norm.pdf(x_range, serie_crystallizer3.mean(), serie_crystallizer3.std())
y_normal_scaled = y_normal * len(serie_crystallizer3) * (serie_crystallizer3.max() - serie_crystallizer3.min()) / 50

fig_normalidade_crystallizer3.add_trace(go.Histogram(
    x=serie_crystallizer3, nbinsx=50,
    marker_color='steelblue', opacity=0.6,
    name='Dados observados'
), row=1, col=2)

fig_normalidade_crystallizer3.add_trace(go.Scatter(
    x=x_range, y=y_normal_scaled,
    mode='lines',
    line=dict(color='red', width=2),
    name='Normal teórica'
), row=1, col=2)

fig_normalidade_crystallizer3.update_xaxes(title_text="Quantis teóricos", row=1, col=1)
fig_normalidade_crystallizer3.update_yaxes(title_text="Quantis observados", row=1, col=1)
fig_normalidade_crystallizer3.update_xaxes(title_text="Resultado de Ferro (ppm)", row=1, col=2)
fig_normalidade_crystallizer3.update_yaxes(title_text="Contagem", row=1, col=2)

fig_normalidade_crystallizer3.update_layout(
    height=500,
    template='plotly_white',
    title="Análise de Normalidade — Resultado de Ferro (ppm) Cristallyzer #3"
)
fig_normalidade_crystallizer3.show()

### Análise sem outliers (Método IQR)

In [ ]:
# Remove outliers via IQR
Q1 = serie_crystallizer3.quantile(0.25)
Q3 = serie_crystallizer3.quantile(0.75)
IQR = Q3 - Q1
serie_iqr_crystallizer3 = serie_crystallizer3[(serie_crystallizer3 >= Q1 - 1.5 * IQR) & (serie_crystallizer3 <= Q3 + 1.5 * IQR)]

print(f"Amostras originais:    {len(serie_crystallizer3)}")
print(f"Amostras sem outliers: {len(serie_iqr_crystallizer3)} ({len(serie_crystallizer3) - len(serie_iqr_crystallizer3)} removidas)")

In [ ]:
descritivas = pd.DataFrame({
    "Métrica": [
        "Contagem", "Média", "Mediana", "Desvio Padrão", "Variância",
        "Mínimo", "Máximo", "Amplitude",
        "Q1 (25%)", "Q3 (75%)", "IQR",
        "Assimetria", "Curtose"
    ],
    "Valor": [
        serie_iqr_crystallizer3.count(), serie_iqr_crystallizer3.mean(), serie_iqr_crystallizer3.median(), serie_iqr_crystallizer3.std(), serie_iqr_crystallizer3.var(),
        serie_iqr_crystallizer3.min(), serie_iqr_crystallizer3.max(), serie_iqr_crystallizer3.max() - serie_iqr_crystallizer3.min(),
        serie_iqr_crystallizer3.quantile(0.25), serie_iqr_crystallizer3.quantile(0.75), serie_iqr_crystallizer3.quantile(0.75) - serie_iqr_crystallizer3.quantile(0.25),
        serie_iqr_crystallizer3.skew(), serie_iqr_crystallizer3.kurt()
    ]
})

descritivas["Valor"] = descritivas["Valor"].round(4)
descritivas.set_index("Métrica", inplace=True)
descritivas.T

#### Histograma sem outliers

In [ ]:
fig_iqr_crystallizer3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Histograma + KDE", "Violin Plot"]
)

fig_iqr_crystallizer3.add_trace(go.Histogram(
    x=serie_iqr_crystallizer3,
    nbinsx=62,
    name="Histograma",
    marker_color='steelblue',
    opacity=0.7
), row=1, col=1)

kde = gaussian_kde(serie_iqr_crystallizer3)
x_range = np.linspace(serie_iqr_crystallizer3.min(), serie_iqr_crystallizer3.max(), 1000)
y_kde = kde(x_range)
y_kde_scaled = y_kde * len(serie_iqr_crystallizer3) * (serie_iqr_crystallizer3.max() - serie_iqr_crystallizer3.min()) / 70

fig_iqr_crystallizer3.add_trace(go.Scatter(
    x=x_range,
    y=y_kde_scaled,
    mode='lines',
    line=dict(color='darkblue', width=2),
    name="KDE"
), row=1, col=1)

fig_iqr_crystallizer3.add_trace(go.Violin(
    y=serie_iqr_crystallizer3,
    name="Violin",
    marker_color='steelblue',
    box_visible=True,
    meanline_visible=True
), row=1, col=2)

fig_iqr_crystallizer3.update_layout(
    height=600,
    template='plotly_white',
    title="Análise Descritiva sem Outliers — Resultado de Ferro (ppm) Cristallyzer #3",
    showlegend=False
)
fig_iqr_crystallizer3.show()

#### Teste de normalidade sem outliers

In [ ]:
# Visualização
fig_iqr_normalidade_crystallizer3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Q-Q Plot", "Histograma + Distribuição Normal Teórica"]
)

# Q-Q Plot
qq = stats.probplot(serie_iqr_crystallizer3, dist="norm")
qq_x = [qq[0][0][0], qq[0][0][-1]]
qq_y = [qq[1][1] + qq[1][0] * qq[0][0][0],
        qq[1][1] + qq[1][0] * qq[0][0][-1]]

fig_iqr_normalidade_crystallizer3.add_trace(go.Scatter(
    x=qq[0][0], y=qq[0][1],
    mode='markers',
    marker=dict(color='steelblue', size=4, opacity=0.5),
    name='Quantis observados'
), row=1, col=1)

fig_iqr_normalidade_crystallizer3.add_trace(go.Scatter(
    x=qq_x, y=qq_y,
    mode='lines',
    line=dict(color='red', width=2),
    name='Linha normal teórica'
), row=1, col=1)

# Histograma + curva normal teórica
x_range = np.linspace(serie_iqr_crystallizer3.min(), serie_iqr_crystallizer3.max(), 300)
y_normal = stats.norm.pdf(x_range, serie_iqr_crystallizer3.mean(), serie_iqr_crystallizer3.std())
y_normal_scaled = y_normal * len(serie_iqr_crystallizer3) * (serie_iqr_crystallizer3.max() - serie_iqr_crystallizer3.min()) / 50

fig_iqr_normalidade_crystallizer3.add_trace(go.Histogram(
    x=serie_iqr_crystallizer3, nbinsx=50,
    marker_color='steelblue', opacity=0.6,
    name='Dados observados'
), row=1, col=2)

fig_iqr_normalidade_crystallizer3.add_trace(go.Scatter(
    x=x_range, y=y_normal_scaled,
    mode='lines',
    line=dict(color='red', width=2),
    name='Normal teórica'
), row=1, col=2)

fig_iqr_normalidade_crystallizer3.update_xaxes(title_text="Quantis teóricos", row=1, col=1)
fig_iqr_normalidade_crystallizer3.update_yaxes(title_text="Quantis observados", row=1, col=1)
fig_iqr_normalidade_crystallizer3.update_xaxes(title_text="Resultado de Ferro (ppm)", row=1, col=2)
fig_iqr_normalidade_crystallizer3.update_yaxes(title_text="Contagem", row=1, col=2)

fig_iqr_normalidade_crystallizer3.update_layout(
    height=500,
    template='plotly_white',
    title="Análise de Normalidade sem outliers — Resultado de Ferro (ppm) Cristallyzer #3"
)
fig_iqr_normalidade_crystallizer3.show()

# Estatísticas descritivas dos eventos

## Crystallizer #1

### KDE dos eventos em Janela Fixa

In [ ]:
DIAS_JANELA = 10
fig = go.Figure()

for _, evento in df_eventos_crystallizer1.iterrows():
    ts = evento["TIMESTAMP"]
    inicio = ts - pd.Timedelta(days=DIAS_JANELA)
    
    mask = (df_crystallizer1['TIMESTAMP'] >= inicio) & (df_crystallizer1['TIMESTAMP'] < ts)
    df_window = df_crystallizer1[mask]
    
    if df_window.empty or df_window["Resultado de Ferro (ppm)"].dropna().shape[0] < 2:
        continue
    
    valores = df_window["Resultado de Ferro (ppm)"].dropna().values
    kde = gaussian_kde(valores)
    x_range = np.linspace(valores.min(), valores.max(), 200)
    y_kde = kde(x_range)
    
    fig.add_trace(go.Scatter(
        x=x_range,
        y=y_kde,
        mode='lines',
        line=dict(width=2),
        fill='tozeroy',
        opacity=0.6,
        name=str(ts.date())
    ))

fig.update_layout(
    template='plotly_white',
    title=f"KDE por Evento (janela de {DIAS_JANELA} dias) - Crystallizer #1",
    xaxis_title="Resultado de Ferro (ppm)",
    yaxis_title="Densidade"
)
fig.show()

### KDE dos eventos em Janela Variável

In [ ]:
def hex_to_rgba(cor, alpha=0.15):
    if cor.startswith('#'):
        rgb = pc.hex_to_rgb(cor)
    else:
        rgb = tuple(int(x) for x in cor.replace('rgb(','').replace(')','').split(','))
    return f'rgba({rgb[0]},{rgb[1]},{rgb[2]},{alpha})'

JANELAS = [15, 12, 9, 6, 3]
eventos_unicos = [str(ts.date()) for ts in df_eventos_crystallizer1["TIMESTAMP"]]
paleta = px.colors.qualitative.Plotly
cor_evento = {nome: paleta[i % len(paleta)] for i, nome in enumerate(eventos_unicos)}

fig = make_subplots(
    rows=len(JANELAS), cols=1,
    subplot_titles=[f"Janela de {d} dias" for d in JANELAS]
)

for row_idx, DIAS_JANELA in enumerate(JANELAS, start=1):
    for _, evento in df_eventos_crystallizer1.iterrows():
        ts = evento["TIMESTAMP"]
        nome = str(ts.date())
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)

        mask = (df_crystallizer1['TIMESTAMP'] >= inicio) & (df_crystallizer1['TIMESTAMP'] < ts)
        df_window = df_crystallizer1[mask]

        if df_window.empty or df_window["Resultado de Ferro (ppm)"].dropna().shape[0] < 2:
            continue

        valores = df_window["Resultado de Ferro (ppm)"].dropna().values
        kde = gaussian_kde(valores)
        x_range = np.linspace(valores.min(), valores.max(), 200)
        y_kde = kde(x_range)

        media   = np.mean(valores)
        mediana = np.median(valores)
        kde_media   = kde(media)[0]
        kde_mediana = kde(mediana)[0]

        cor      = cor_evento[nome]
        cor_fill = hex_to_rgba(cor, alpha=0.15)  # transparência aqui

        # KDE
        fig.add_trace(go.Scatter(
            x=x_range, y=y_kde,
            mode='lines',
            line=dict(color=cor, width=2),
            fill='tozeroy',
            fillcolor=cor_fill,
            name=nome,
            legendgroup=nome,
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

        # Linha vertical da média
        fig.add_trace(go.Scatter(
            x=[media, media], y=[0, kde_media],
            mode='lines',
            line=dict(color=cor, width=1.5, dash='dash'),
            legendgroup=nome, showlegend=False
        ), row=row_idx, col=1)

        # Marcador da média
        fig.add_trace(go.Scatter(
            x=[media], y=[kde_media],
            mode='markers+text',
            marker=dict(color=cor, size=8, symbol='circle'),
            text=[f"μ={media:.2f}"],
            textposition='top center',
            textfont=dict(size=9),
            legendgroup=nome, showlegend=False
        ), row=row_idx, col=1)

        # Linha vertical da mediana
        fig.add_trace(go.Scatter(
            x=[mediana, mediana], y=[0, kde_mediana],
            mode='lines',
            line=dict(color=cor, width=1.5, dash='dot'),
            legendgroup=nome, showlegend=False
        ), row=row_idx, col=1)

        # Marcador da mediana
        fig.add_trace(go.Scatter(
            x=[mediana], y=[kde_mediana],
            mode='markers+text',
            marker=dict(color=cor, size=8, symbol='diamond'),
            text=[f"med={mediana:.2f}"],
            textposition='top center',
            textfont=dict(size=9),
            legendgroup=nome, showlegend=False
        ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Densidade", row=row_idx, col=1)
    fig.update_xaxes(title_text="Resultado de Ferro (ppm)", row=row_idx, col=1)

fig.update_layout(
    height=400 * len(JANELAS),
    template='plotly_white',
    title="KDE por Evento — Diferentes Janelas Temporais - Crystallizer #1"
)
fig.show()

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

fig = go.Figure()

for DIAS_JANELA in JANELAS:
    for classe in [0, 1]:
        valores_classe = []
        for _, evento in df_eventos_crystallizer1.iterrows():
            if int(evento["Real"]) != classe:
                continue
            ts     = evento["TIMESTAMP"]
            inicio = ts - pd.Timedelta(days=DIAS_JANELA)
            mask   = (
                (df_crystallizer1['TIMESTAMP'] >= inicio) &
                (df_crystallizer1['TIMESTAMP'] <  ts)
            )
            v = df_crystallizer1[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(v) >= 2:
                valores_classe.extend(v.tolist())

        fig.add_trace(go.Violin(
            y=valores_classe,
            x=[f"{DIAS_JANELA}d"] * len(valores_classe),
            name=nomes_classe[classe],
            legendgroup=nomes_classe[classe],
            showlegend=(DIAS_JANELA == JANELAS[0]),
            side="negative" if classe == 0 else "positive",
            line_color=cores_classe[classe],
            meanline_visible=True,
            points=False
        ))

fig.update_layout(
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal - Crystallizer #1",
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    yaxis_title="Fe (ppm)",
    xaxis_title="Janela",
    height=500
)
fig.show()

### Análise de Estatísticas por média móvel
Analisando 15 dias anteriores ao evento com média móvel de 3 dias

In [ ]:
eventos_unicos = [str(ts.date()) for ts in df_eventos_crystallizer1["TIMESTAMP"]]
paleta = px.colors.qualitative.Plotly
cor_evento = {nome: paleta[i % len(paleta)] for i, nome in enumerate(eventos_unicos)}

DIAS_TOTAL = 16
DIAS_ROLL  = 3

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=["Média móvel (ppm)", "Mediana móvel (ppm)", "Desvio Padrão móvel"],
    shared_xaxes=True
)

for _, evento in df_eventos_crystallizer1.iterrows():
    ts    = evento["TIMESTAMP"]
    nome  = str(ts.date())
    cor   = cor_evento.get(nome, "#888888")
    inicio = ts - pd.Timedelta(days=DIAS_TOTAL)

    mask = (
        (df_crystallizer1['TIMESTAMP'] >= inicio) &
        (df_crystallizer1['TIMESTAMP'] <= ts)
    )
    df_ev = df_crystallizer1[mask].copy()
    if df_ev.empty:
        continue

    # Dias relativos ao evento (negativo = antes)
    df_ev["dias_rel"] = (df_ev["TIMESTAMP"] - ts).dt.total_seconds() / 86400

    # Resample para grade diária e aplica janela deslizante
    df_ev = df_ev.set_index("TIMESTAMP")
    ppm   = df_ev["Resultado de Ferro (ppm)"].dropna()

    roll_mean = ppm.rolling(f"{DIAS_ROLL}D").mean()
    roll_med  = ppm.rolling(f"{DIAS_ROLL}D").median()
    roll_std  = ppm.rolling(f"{DIAS_ROLL}D").std()

    dias = (ppm.index - ts).total_seconds() / 86400

    for row_idx, (stat, label) in enumerate(
        zip([roll_mean, roll_med, roll_std], ["Média", "Mediana", "Std"]), start=1
    ):
        fig.add_trace(go.Scatter(
            x=dias, y=stat.values,
            mode="lines", name=nome,
            line=dict(color=cor, width=1.5),
            legendgroup=nome,
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

    # Linha vertical no evento
    for row_idx in range(1, 4):
        fig.add_shape(
            type="line",
            x0=0, x1=0, y0=0, y1=1,
            yref="paper",
            xref=f"x{row_idx}" if row_idx > 1 else "x",
            line=dict(color="black", dash="dash", width=1),
            row=row_idx, col=1
        )

fig.update_xaxes(title_text="Dias antes do evento", row=3, col=1)
fig.update_layout(
    title="Evolução Temporal das Estatísticas — 15 dias antes do evento - Cystallizer #1",
    template="plotly_white",
    height=700
)
fig.show()

### Análise por BoxPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
fig = make_subplots(
    rows=1, cols=len(JANELAS),
    subplot_titles=[f"{d} dias" for d in JANELAS]
)

cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

for col_idx, DIAS_JANELA in enumerate(JANELAS, start=1):
    dados_por_classe = {0: [], 1: []}

    for _, evento in df_eventos_crystallizer1.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df_crystallizer1['TIMESTAMP'] >= inicio) &
            (df_crystallizer1['TIMESTAMP'] <  ts)
        )
        valores = df_crystallizer1[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(valores) < 2:
            continue
        classe = int(evento["Real"])
        dados_por_classe[classe].extend(valores.tolist())

    for classe, valores in dados_por_classe.items():
        fig.add_trace(go.Box(
            y=valores,
            name=nomes_classe[classe],
            marker_color=cores_classe[classe],
            legendgroup=nomes_classe[classe],
            showlegend=(col_idx == 1),
            boxmean='sd',        # mostra média + desvio além da mediana
        ), row=1, col=col_idx)

fig.update_layout(
    title="Distribuição Fe (ppm): Real vs Falso Positivo por Janela - Crystallizer #1",
    template="plotly_white",
    height=500,
    boxmode="group"
)
fig.show()

## Crystallizer #2

### KDE dos eventos em Janela Fixa

In [ ]:
DIAS_JANELA = 10
fig = go.Figure()

for _, evento in df_eventos_crystallizer2.iterrows():
    ts = evento["TIMESTAMP"]
    inicio = ts - pd.Timedelta(days=DIAS_JANELA)
    
    mask = (df_crystallizer2['TIMESTAMP'] >= inicio) & (df_crystallizer2['TIMESTAMP'] < ts)
    df_window = df_crystallizer2[mask]
    
    if df_window.empty or df_window["Resultado de Ferro (ppm)"].dropna().shape[0] < 2:
        continue
    
    valores = df_window["Resultado de Ferro (ppm)"].dropna().values
    kde = gaussian_kde(valores)
    x_range = np.linspace(valores.min(), valores.max(), 200)
    y_kde = kde(x_range)
    
    fig.add_trace(go.Scatter(
        x=x_range,
        y=y_kde,
        mode='lines',
        line=dict(width=2),
        fill='tozeroy',
        opacity=0.6,
        name=str(ts.date())
    ))

fig.update_layout(
    template='plotly_white',
    title=f"KDE por Evento (janela de {DIAS_JANELA} dias) - Crystallizer #2",
    xaxis_title="Resultado de Ferro (ppm)",
    yaxis_title="Densidade"
)
fig.show()

### KDE dos eventos em Janela Variável

In [ ]:
def hex_to_rgba(cor, alpha=0.15):
    if cor.startswith('#'):
        rgb = pc.hex_to_rgb(cor)
    else:
        rgb = tuple(int(x) for x in cor.replace('rgb(','').replace(')','').split(','))
    return f'rgba({rgb[0]},{rgb[1]},{rgb[2]},{alpha})'

JANELAS = [15, 12, 9, 6, 3]
eventos_unicos = [str(ts.date()) for ts in df_eventos_crystallizer2["TIMESTAMP"]]
paleta = px.colors.qualitative.Plotly
cor_evento = {nome: paleta[i % len(paleta)] for i, nome in enumerate(eventos_unicos)}

fig = make_subplots(
    rows=len(JANELAS), cols=1,
    subplot_titles=[f"Janela de {d} dias" for d in JANELAS]
)

for row_idx, DIAS_JANELA in enumerate(JANELAS, start=1):
    for _, evento in df_eventos_crystallizer2.iterrows():
        ts = evento["TIMESTAMP"]
        nome = str(ts.date())
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)

        mask = (df_crystallizer2['TIMESTAMP'] >= inicio) & (df_crystallizer2['TIMESTAMP'] < ts)
        df_window = df_crystallizer2[mask]

        if df_window.empty or df_window["Resultado de Ferro (ppm)"].dropna().shape[0] < 2:
            continue

        valores = df_window["Resultado de Ferro (ppm)"].dropna().values
        kde = gaussian_kde(valores)
        x_range = np.linspace(valores.min(), valores.max(), 200)
        y_kde = kde(x_range)

        media   = np.mean(valores)
        mediana = np.median(valores)
        kde_media   = kde(media)[0]
        kde_mediana = kde(mediana)[0]

        cor      = cor_evento[nome]
        cor_fill = hex_to_rgba(cor, alpha=0.15)  # transparência aqui

        # KDE
        fig.add_trace(go.Scatter(
            x=x_range, y=y_kde,
            mode='lines',
            line=dict(color=cor, width=2),
            fill='tozeroy',
            fillcolor=cor_fill,
            name=nome,
            legendgroup=nome,
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

        # Linha vertical da média
        fig.add_trace(go.Scatter(
            x=[media, media], y=[0, kde_media],
            mode='lines',
            line=dict(color=cor, width=1.5, dash='dash'),
            legendgroup=nome, showlegend=False
        ), row=row_idx, col=1)

        # Marcador da média
        fig.add_trace(go.Scatter(
            x=[media], y=[kde_media],
            mode='markers+text',
            marker=dict(color=cor, size=8, symbol='circle'),
            text=[f"μ={media:.2f}"],
            textposition='top center',
            textfont=dict(size=9),
            legendgroup=nome, showlegend=False
        ), row=row_idx, col=1)

        # Linha vertical da mediana
        fig.add_trace(go.Scatter(
            x=[mediana, mediana], y=[0, kde_mediana],
            mode='lines',
            line=dict(color=cor, width=1.5, dash='dot'),
            legendgroup=nome, showlegend=False
        ), row=row_idx, col=1)

        # Marcador da mediana
        fig.add_trace(go.Scatter(
            x=[mediana], y=[kde_mediana],
            mode='markers+text',
            marker=dict(color=cor, size=8, symbol='diamond'),
            text=[f"med={mediana:.2f}"],
            textposition='top center',
            textfont=dict(size=9),
            legendgroup=nome, showlegend=False
        ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Densidade", row=row_idx, col=1)
    fig.update_xaxes(title_text="Resultado de Ferro (ppm)", row=row_idx, col=1)

fig.update_layout(
    height=400 * len(JANELAS),
    template='plotly_white',
    title="KDE por Evento — Diferentes Janelas Temporais - Crystallizer #2"
)
fig.show()

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

fig = go.Figure()

for DIAS_JANELA in JANELAS:
    for classe in [0, 1]:
        valores_classe = []
        for _, evento in df_eventos_crystallizer2.iterrows():
            if int(evento["Real"]) != classe:
                continue
            ts     = evento["TIMESTAMP"]
            inicio = ts - pd.Timedelta(days=DIAS_JANELA)
            mask   = (
                (df_crystallizer2['TIMESTAMP'] >= inicio) &
                (df_crystallizer2['TIMESTAMP'] <  ts)
            )
            v = df_crystallizer2[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(v) >= 2:
                valores_classe.extend(v.tolist())

        fig.add_trace(go.Violin(
            y=valores_classe,
            x=[f"{DIAS_JANELA}d"] * len(valores_classe),
            name=nomes_classe[classe],
            legendgroup=nomes_classe[classe],
            showlegend=(DIAS_JANELA == JANELAS[0]),
            side="negative" if classe == 0 else "positive",
            line_color=cores_classe[classe],
            meanline_visible=True,
            points=False
        ))

fig.update_layout(
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal - Crystallizer #2",
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    yaxis_title="Fe (ppm)",
    xaxis_title="Janela",
    height=500
)
fig.show()

### Análise de Estatísticas por média móvel
Analisando 15 dias anteriores ao evento com média móvel de 3 dias

In [ ]:
eventos_unicos = [str(ts.date()) for ts in df_eventos_crystallizer2["TIMESTAMP"]]
paleta = px.colors.qualitative.Plotly
cor_evento = {nome: paleta[i % len(paleta)] for i, nome in enumerate(eventos_unicos)}

DIAS_TOTAL = 16
DIAS_ROLL  = 3

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=["Média móvel (ppm)", "Mediana móvel (ppm)", "Desvio Padrão móvel"],
    shared_xaxes=True
)

for _, evento in df_eventos_crystallizer2.iterrows():
    ts    = evento["TIMESTAMP"]
    nome  = str(ts.date())
    cor   = cor_evento.get(nome, "#888888")
    inicio = ts - pd.Timedelta(days=DIAS_TOTAL)

    mask = (
        (df_crystallizer2['TIMESTAMP'] >= inicio) &
        (df_crystallizer2['TIMESTAMP'] <= ts)
    )
    df_ev = df_crystallizer2[mask].copy()
    if df_ev.empty:
        continue

    # Dias relativos ao evento (negativo = antes)
    df_ev["dias_rel"] = (df_ev["TIMESTAMP"] - ts).dt.total_seconds() / 86400

    # Resample para grade diária e aplica janela deslizante
    df_ev = df_ev.set_index("TIMESTAMP")
    ppm   = df_ev["Resultado de Ferro (ppm)"].dropna()

    roll_mean = ppm.rolling(f"{DIAS_ROLL}D").mean()
    roll_med  = ppm.rolling(f"{DIAS_ROLL}D").median()
    roll_std  = ppm.rolling(f"{DIAS_ROLL}D").std()

    dias = (ppm.index - ts).total_seconds() / 86400

    for row_idx, (stat, label) in enumerate(
        zip([roll_mean, roll_med, roll_std], ["Média", "Mediana", "Std"]), start=1
    ):
        fig.add_trace(go.Scatter(
            x=dias, y=stat.values,
            mode="lines", name=nome,
            line=dict(color=cor, width=1.5),
            legendgroup=nome,
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

    # Linha vertical no evento
    for row_idx in range(1, 4):
        fig.add_shape(
            type="line",
            x0=0, x1=0, y0=0, y1=1,
            yref="paper",
            xref=f"x{row_idx}" if row_idx > 1 else "x",
            line=dict(color="black", dash="dash", width=1),
            row=row_idx, col=1
        )

fig.update_xaxes(title_text="Dias antes do evento", row=3, col=1)
fig.update_layout(
    title="Evolução Temporal das Estatísticas — 15 dias antes do evento - Crystallizer #2",
    template="plotly_white",
    height=700
)
fig.show()

### Análise por BoxPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
fig = make_subplots(
    rows=1, cols=len(JANELAS),
    subplot_titles=[f"{d} dias" for d in JANELAS]
)

cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

for col_idx, DIAS_JANELA in enumerate(JANELAS, start=1):
    dados_por_classe = {0: [], 1: []}

    for _, evento in df_eventos_crystallizer2.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df_crystallizer2['TIMESTAMP'] >= inicio) &
            (df_crystallizer2['TIMESTAMP'] <  ts)
        )
        valores = df_crystallizer2[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(valores) < 2:
            continue
        classe = int(evento["Real"])
        dados_por_classe[classe].extend(valores.tolist())

    for classe, valores in dados_por_classe.items():
        fig.add_trace(go.Box(
            y=valores,
            name=nomes_classe[classe],
            marker_color=cores_classe[classe],
            legendgroup=nomes_classe[classe],
            showlegend=(col_idx == 1),
            boxmean='sd',        # mostra média + desvio além da mediana
        ), row=1, col=col_idx)

fig.update_layout(
    title="Distribuição Fe (ppm): Real vs Falso Positivo por Janela - Crystallizer #2",
    template="plotly_white",
    height=500,
    boxmode="group"
)
fig.show()

## Crystallizer #3

### KDE dos eventos em Janela Fixa

In [ ]:
DIAS_JANELA = 10
fig = go.Figure()

for _, evento in df_eventos_crystallizer3.iterrows():
    ts = evento["TIMESTAMP"]
    inicio = ts - pd.Timedelta(days=DIAS_JANELA)
    
    mask = (df_crystallizer3['TIMESTAMP'] >= inicio) & (df_crystallizer3['TIMESTAMP'] < ts)
    df_window = df_crystallizer3[mask]
    
    if df_window.empty or df_window["Resultado de Ferro (ppm)"].dropna().shape[0] < 2:
        continue
    
    valores = df_window["Resultado de Ferro (ppm)"].dropna().values
    kde = gaussian_kde(valores)
    x_range = np.linspace(valores.min(), valores.max(), 200)
    y_kde = kde(x_range)
    
    fig.add_trace(go.Scatter(
        x=x_range,
        y=y_kde,
        mode='lines',
        line=dict(width=2),
        fill='tozeroy',
        opacity=0.6,
        name=str(ts.date())
    ))

fig.update_layout(
    template='plotly_white',
    title=f"KDE por Evento (janela de {DIAS_JANELA} dias) - Crystallizer #3",
    xaxis_title="Resultado de Ferro (ppm)",
    yaxis_title="Densidade"
)
fig.show()

### KDE dos eventos em Janela Variável

In [ ]:
def hex_to_rgba(cor, alpha=0.15):
    if cor.startswith('#'):
        rgb = pc.hex_to_rgb(cor)
    else:
        rgb = tuple(int(x) for x in cor.replace('rgb(','').replace(')','').split(','))
    return f'rgba({rgb[0]},{rgb[1]},{rgb[2]},{alpha})'

JANELAS = [15, 12, 9, 6, 3]
eventos_unicos = [str(ts.date()) for ts in df_eventos_crystallizer3["TIMESTAMP"]]
paleta = px.colors.qualitative.Plotly
cor_evento = {nome: paleta[i % len(paleta)] for i, nome in enumerate(eventos_unicos)}

fig = make_subplots(
    rows=len(JANELAS), cols=1,
    subplot_titles=[f"Janela de {d} dias" for d in JANELAS]
)

for row_idx, DIAS_JANELA in enumerate(JANELAS, start=1):
    for _, evento in df_eventos_crystallizer3.iterrows():
        ts = evento["TIMESTAMP"]
        nome = str(ts.date())
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)

        mask = (df_crystallizer3['TIMESTAMP'] >= inicio) & (df_crystallizer3['TIMESTAMP'] < ts)
        df_window = df_crystallizer3[mask]

        if df_window.empty or df_window["Resultado de Ferro (ppm)"].dropna().shape[0] < 2:
            continue

        valores = df_window["Resultado de Ferro (ppm)"].dropna().values
        kde = gaussian_kde(valores)
        x_range = np.linspace(valores.min(), valores.max(), 200)
        y_kde = kde(x_range)

        media   = np.mean(valores)
        mediana = np.median(valores)
        kde_media   = kde(media)[0]
        kde_mediana = kde(mediana)[0]

        cor      = cor_evento[nome]
        cor_fill = hex_to_rgba(cor, alpha=0.15)  # transparência aqui

        # KDE
        fig.add_trace(go.Scatter(
            x=x_range, y=y_kde,
            mode='lines',
            line=dict(color=cor, width=2),
            fill='tozeroy',
            fillcolor=cor_fill,
            name=nome,
            legendgroup=nome,
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

        # Linha vertical da média
        fig.add_trace(go.Scatter(
            x=[media, media], y=[0, kde_media],
            mode='lines',
            line=dict(color=cor, width=1.5, dash='dash'),
            legendgroup=nome, showlegend=False
        ), row=row_idx, col=1)

        # Marcador da média
        fig.add_trace(go.Scatter(
            x=[media], y=[kde_media],
            mode='markers+text',
            marker=dict(color=cor, size=8, symbol='circle'),
            text=[f"μ={media:.2f}"],
            textposition='top center',
            textfont=dict(size=9),
            legendgroup=nome, showlegend=False
        ), row=row_idx, col=1)

        # Linha vertical da mediana
        fig.add_trace(go.Scatter(
            x=[mediana, mediana], y=[0, kde_mediana],
            mode='lines',
            line=dict(color=cor, width=1.5, dash='dot'),
            legendgroup=nome, showlegend=False
        ), row=row_idx, col=1)

        # Marcador da mediana
        fig.add_trace(go.Scatter(
            x=[mediana], y=[kde_mediana],
            mode='markers+text',
            marker=dict(color=cor, size=8, symbol='diamond'),
            text=[f"med={mediana:.2f}"],
            textposition='top center',
            textfont=dict(size=9),
            legendgroup=nome, showlegend=False
        ), row=row_idx, col=1)

    fig.update_yaxes(title_text="Densidade", row=row_idx, col=1)
    fig.update_xaxes(title_text="Resultado de Ferro (ppm)", row=row_idx, col=1)

fig.update_layout(
    height=400 * len(JANELAS),
    template='plotly_white',
    title="KDE por Evento — Diferentes Janelas Temporais - Crystallizer #3"
)
fig.show()

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]
cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

fig = go.Figure()

for DIAS_JANELA in JANELAS:
    for classe in [0, 1]:
        valores_classe = []
        for _, evento in df_eventos_crystallizer3.iterrows():
            if int(evento["Real"]) != classe:
                continue
            ts     = evento["TIMESTAMP"]
            inicio = ts - pd.Timedelta(days=DIAS_JANELA)
            mask   = (
                (df_crystallizer3['TIMESTAMP'] >= inicio) &
                (df_crystallizer3['TIMESTAMP'] <  ts)
            )
            v = df_crystallizer3[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(v) >= 2:
                valores_classe.extend(v.tolist())

        fig.add_trace(go.Violin(
            y=valores_classe,
            x=[f"{DIAS_JANELA}d"] * len(valores_classe),
            name=nomes_classe[classe],
            legendgroup=nomes_classe[classe],
            showlegend=(DIAS_JANELA == JANELAS[0]),
            side="negative" if classe == 0 else "positive",
            line_color=cores_classe[classe],
            meanline_visible=True,
            points=False
        ))

fig.update_layout(
    title="Violin Plot: Real vs Falso Positivo por Janela Temporal - Crystallizer #3",
    template="plotly_white",
    violingap=0.05,
    violinmode="overlay",
    yaxis_title="Fe (ppm)",
    xaxis_title="Janela",
    height=500
)
fig.show()

### Análise de Estatísticas por média móvel
Analisando 15 dias anteriores ao evento com média móvel de 3 dias

In [ ]:
eventos_unicos = [str(ts.date()) for ts in df_eventos_crystallizer3["TIMESTAMP"]]
paleta = px.colors.qualitative.Plotly
cor_evento = {nome: paleta[i % len(paleta)] for i, nome in enumerate(eventos_unicos)}

DIAS_TOTAL = 16
DIAS_ROLL  = 3

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=["Média móvel (ppm)", "Mediana móvel (ppm)", "Desvio Padrão móvel"],
    shared_xaxes=True
)

for _, evento in df_eventos_crystallizer3.iterrows():
    ts    = evento["TIMESTAMP"]
    nome  = str(ts.date())
    cor   = cor_evento.get(nome, "#888888")
    inicio = ts - pd.Timedelta(days=DIAS_TOTAL)

    mask = (
        (df_crystallizer3['TIMESTAMP'] >= inicio) &
        (df_crystallizer3['TIMESTAMP'] <= ts)
    )
    df_ev = df_crystallizer3[mask].copy()
    if df_ev.empty:
        continue

    # Dias relativos ao evento (negativo = antes)
    df_ev["dias_rel"] = (df_ev["TIMESTAMP"] - ts).dt.total_seconds() / 86400

    # Resample para grade diária e aplica janela deslizante
    df_ev = df_ev.set_index("TIMESTAMP")
    ppm   = df_ev["Resultado de Ferro (ppm)"].dropna()

    roll_mean = ppm.rolling(f"{DIAS_ROLL}D").mean()
    roll_med  = ppm.rolling(f"{DIAS_ROLL}D").median()
    roll_std  = ppm.rolling(f"{DIAS_ROLL}D").std()

    dias = (ppm.index - ts).total_seconds() / 86400

    for row_idx, (stat, label) in enumerate(
        zip([roll_mean, roll_med, roll_std], ["Média", "Mediana", "Std"]), start=1
    ):
        fig.add_trace(go.Scatter(
            x=dias, y=stat.values,
            mode="lines", name=nome,
            line=dict(color=cor, width=1.5),
            legendgroup=nome,
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

    # Linha vertical no evento
    for row_idx in range(1, 4):
        fig.add_shape(
            type="line",
            x0=0, x1=0, y0=0, y1=1,
            yref="paper",
            xref=f"x{row_idx}" if row_idx > 1 else "x",
            line=dict(color="black", dash="dash", width=1),
            row=row_idx, col=1
        )

fig.update_xaxes(title_text="Dias antes do evento", row=3, col=1)
fig.update_layout(
    title="Evolução Temporal das Estatísticas — 15 dias antes do evento - Crystallizer #2",
    template="plotly_white",
    height=700
)
fig.show()

### Análise por BoxPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
fig = make_subplots(
    rows=1, cols=len(JANELAS),
    subplot_titles=[f"{d} dias" for d in JANELAS]
)

cores_classe = {0: "#4878CF", 1: "#D65F5F"}
nomes_classe = {0: "Falso Positivo", 1: "Real"}

for col_idx, DIAS_JANELA in enumerate(JANELAS, start=1):
    dados_por_classe = {0: [], 1: []}

    for _, evento in df_eventos_crystallizer3.iterrows():
        ts     = evento["TIMESTAMP"]
        inicio = ts - pd.Timedelta(days=DIAS_JANELA)
        mask   = (
            (df_crystallizer3['TIMESTAMP'] >= inicio) &
            (df_crystallizer3['TIMESTAMP'] <  ts)
        )
        valores = df_crystallizer3[mask]["Resultado de Ferro (ppm)"].dropna().values
        if len(valores) < 2:
            continue
        classe = int(evento["Real"])
        dados_por_classe[classe].extend(valores.tolist())

    for classe, valores in dados_por_classe.items():
        fig.add_trace(go.Box(
            y=valores,
            name=nomes_classe[classe],
            marker_color=cores_classe[classe],
            legendgroup=nomes_classe[classe],
            showlegend=(col_idx == 1),
            boxmean='sd',        # mostra média + desvio além da mediana
        ), row=1, col=col_idx)

fig.update_layout(
    title="Distribuição Fe (ppm): Real vs Falso Positivo por Janela - Crystallizer #2",
    template="plotly_white",
    height=500,
    boxmode="group"
)
fig.show()

# Clusterização (Não supervisionada)

### Extração de Features

In [ ]:
def extract_features(df_window, df_baseline=None):
    """
    Extrai características da janela para detecção de anomalias.

    Parâmetros
    ----------
    df_window   : DataFrame com colunas TIMESTAMP e 'Resultado de Ferro (ppm)'
                  contendo os dados da janela anterior ao evento.
    df_baseline : DataFrame com o mesmo schema, contendo o período de
                  referência histórica (DIAS_BASELINE dias antes da janela).
                  Se None, as features de contexto histórico recebem NaN.
    """
    if len(df_window) < 5:
        return None

    #  Vetores base 
    t_hours = (df_window['TIMESTAMP'] - df_window['TIMESTAMP'].min()) \
                .dt.total_seconds() / 3600.0
    y_ppm   = df_window['Resultado de Ferro (ppm)'].values

    #  Derivadas — primeira (taxa ppm/hora) 
    dt    = np.diff(t_hours)
    dy    = np.diff(y_ppm)
    rates = np.divide(dy, dt, out=np.zeros_like(dy), where=dt != 0)

    #  Derivadas — segunda (aceleração ppm/hora²) 
    if len(rates) > 1:
        aceleracoes      = np.diff(rates) / (dt[1:] + 1e-9)
        aceleracao_media = float(np.mean(aceleracoes))
        aceleracao_final = float(aceleracoes[-1])
    else:
        aceleracao_media = aceleracao_final = 0.0

    #  Inclinação linear global 
    lr    = LinearRegression().fit(t_hours.values.reshape(-1, 1), y_ppm)
    slope = float(lr.coef_[0])

    #  Integral 
    area = float(trapezoid(y=y_ppm, x=t_hours))

    #  Último movimento antes do evento 
    last_dy   = float(dy[-1]) if len(dy) > 0 else 0.0
    last_dt   = float(dt[-1]) if len(dt) > 0 else 0.0
    ema_final = float(
        df_window['Resultado de Ferro (ppm)']
        .ewm(span=len(df_window), adjust=False).mean().iloc[-1]
    )

    #  Complexidade / inversões de tendência 
    inversoes_tendencia = int(np.sum(np.diff(np.sign(rates)) != 0)) \
                          if len(rates) > 1 else 0

    #  Energia das oscilações 
    # Soma dos quadrados das variações normalizadas — captura magnitude das mudanças
    energia_oscilacao = float(np.sum((dy / (np.std(y_ppm) + 1e-9)) ** 2))

    #  Picos 
    rms          = np.sqrt(np.mean(y_ppm ** 2))
    crest_factor = float(np.max(np.abs(y_ppm)) / rms) if rms > 0 else 0.0

    picos_idx, props = find_peaks(y_ppm, prominence=0)
    max_prominence   = float(np.max(props['prominences'])) if len(picos_idx) > 0 else 0.0

    # Picos com proeminência >= 1 desvio padrão (picos expressivos)
    picos_exp_idx, _ = find_peaks(y_ppm, prominence=float(np.std(y_ppm)))

    #  Comportamento pós-pico 
    # Captura se o sinal sustenta ou cai após o último pico expressivo
    if len(picos_exp_idx) > 0:
        idx_ultimo_pico         = picos_exp_idx[-1]
        y_apos                  = y_ppm[idx_ultimo_pico:]
        tempo_desde_ultimo_pico = float(t_hours.max() - t_hours.values[idx_ultimo_pico])
        media_apos_ultimo_pico  = float(np.mean(y_apos))
        # decay > 0: sinal caiu após o pico | decay < 0: sinal subiu (raro)
        decay_apos_pico         = float(y_apos[0] - y_apos[-1]) if len(y_apos) > 1 else 0.0
    else:
        tempo_desde_ultimo_pico = float(t_hours.max())
        media_apos_ultimo_pico  = float(np.mean(y_ppm))
        decay_apos_pico         = 0.0

    #  Razão pico / vale 
    vales_idx, _ = find_peaks(-y_ppm, prominence=0)
    media_picos  = float(np.mean(y_ppm[picos_idx])) if len(picos_idx) > 0 \
                   else float(np.max(y_ppm))
    media_vales  = float(np.mean(y_ppm[vales_idx])) if len(vales_idx) > 0 \
                   else float(np.min(y_ppm))
    ratio_pico_vale = media_picos / (media_vales + 1e-9)

    #  Shape da distribuição 
    ppm_skewness = float(skew(y_ppm))
    ppm_kurtosis = float(kurtosis(y_ppm))
    range_norm   = float((np.max(y_ppm) - np.min(y_ppm)) / (np.mean(y_ppm) + 1e-9))

    #  Tendência por metades 
    mid                   = len(y_ppm) // 2
    media_primeira_metade = float(np.mean(y_ppm[:mid])) if mid >= 1 else float(np.mean(y_ppm))
    media_segunda_metade  = float(np.mean(y_ppm[mid:])) if mid >= 1 else float(np.mean(y_ppm))
    ratio_metades         = media_segunda_metade / (media_primeira_metade + 1e-9)

    t_second = t_hours.values[mid:]
    y_second = y_ppm[mid:]
    if len(t_second) >= 2:
        slope_recente = float(
            LinearRegression().fit(t_second.reshape(-1, 1), y_second).coef_[0]
        )
    else:
        slope_recente = slope

    #  Tendência no terço final 
    # Mais granular que slope_recente — captura os últimos ~2 dias da janela de 7d
    terco   = len(y_ppm) * 2 // 3
    t_final = t_hours.values[terco:]
    y_final = y_ppm[terco:]
    if len(t_final) >= 2:
        slope_final = float(
            LinearRegression().fit(t_final.reshape(-1, 1), y_final).coef_[0]
        )
        nivel_final = float(np.mean(y_final))
    else:
        slope_final = slope_recente
        nivel_final = float(y_ppm[-1])

    #  Estacionariedade / mudança de patamar 
    # Mede se o sinal migrou para um nível diferente ao longo da janela.
    # > 0: subiu de patamar | < 0: desceu | ~ 0: estável
    terco_n     = max(len(y_ppm) // 3, 1)
    shift_nivel = float(
        (np.mean(y_ppm[2 * terco_n:]) - np.mean(y_ppm[:terco_n]))
        / (np.std(y_ppm) + 1e-9)
    )

    #  Tempo acima do limite operacional fixo 
    pct_acima_limite   = float(np.mean(y_ppm > threshold))
    # Integra o tempo (horas) em que o sinal ficou acima do limite
    acima_flag         = (y_ppm[:-1] > threshold).astype(float)
    horas_acima_limite = float(np.sum(acima_flag * dt))

    #  Regularidade temporal da amostragem 
    n_amostras    = len(y_ppm)
    dt_medio      = float(np.mean(dt))  if len(dt) > 0 else 0.0
    dt_min        = float(np.min(dt))   if len(dt) > 0 else 0.0
    dt_variancia  = float(np.var(dt))   if len(dt) > 1 else 0.0
    duracao_total = float(t_hours.max()) if t_hours.max() > 0 else 1e-9
    freq_amostras = n_amostras / (duracao_total + 1e-9)

    #  Contexto histórico (requer df_baseline) 
    if df_baseline is not None and len(df_baseline) >= 2:
        y_base         = df_baseline['Resultado de Ferro (ppm)'].values
        baseline_media = float(np.mean(y_base))
        baseline_std   = float(np.std(y_base))

        zscore_max          = (np.max(y_ppm)  - baseline_media) / (baseline_std + 1e-9)
        zscore_media        = (np.mean(y_ppm) - baseline_media) / (baseline_std + 1e-9)
        ratio_media         = np.mean(y_ppm) / (baseline_media + 1e-9)
        ratio_max           = np.max(y_ppm)  / (baseline_media + 1e-9)

        threshold_hist      = float(np.percentile(y_base, 75))
        n_acima_threshold   = int(np.sum(y_ppm > threshold_hist))
        pct_acima_threshold = float(np.mean(y_ppm > threshold_hist))

        flags  = np.concatenate([[False], y_ppm > threshold_hist, [False]])
        diffs  = np.diff(flags.astype(int))
        starts = np.where(diffs ==  1)[0]
        ends   = np.where(diffs == -1)[0]
        max_run_acima = int(np.max(ends - starts)) if len(starts) > 0 else 0

    else:
        zscore_max = zscore_media = ratio_media = ratio_max = np.nan
        n_acima_threshold = pct_acima_threshold = max_run_acima = np.nan

    #  Dicionário final 
    return {
        # Taxas de variação (1ª derivada)
        'taxa_max':                 float(np.max(rates)) if len(rates) > 0 else 0.0,
        'taxa_media':               float(np.mean(rates)) if len(rates) > 0 else 0.0,
        # Aceleração (2ª derivada)
        'aceleracao_media':         aceleracao_media,
        'aceleracao_final':         aceleracao_final,
        # Tendência global
        'slope':                    slope,
        'slope_recente':            slope_recente,
        'slope_final':              slope_final,
        'nivel_final':              nivel_final,
        # Estacionariedade
        'shift_nivel':              shift_nivel,
        # Magnitude e acumulação
        'area_curva':               area,
        'ppm_max':                  float(np.max(y_ppm)),
        'ppm_min':                  float(np.min(y_ppm)),
        'ppm_media':                float(np.mean(y_ppm)),
        'ppm_std':                  float(np.std(y_ppm)),
        # Último movimento
        'ultimo_dy':                last_dy,
        'ultimo_dt':                last_dt,
        'ema_final':                ema_final,
        # Complexidade e energia
        'inversoes_tendencia':      inversoes_tendencia,
        'energia_oscilacao':        energia_oscilacao,
        # Picos
        'crest_factor':             crest_factor,
        'max_prominence':           max_prominence,
        'tempo_desde_ultimo_pico':  tempo_desde_ultimo_pico,
        'media_apos_ultimo_pico':   media_apos_ultimo_pico,
        'decay_apos_pico':          decay_apos_pico,
        'ratio_pico_vale':          ratio_pico_vale,
        # Shape da distribuição
        'skewness':                 ppm_skewness,
        'kurtosis':                 ppm_kurtosis,
        'range_norm':               range_norm,
        # Tendência por metades
        'media_primeira_metade':    media_primeira_metade,
        'media_segunda_metade':     media_segunda_metade,
        'ratio_metades':            ratio_metades,
        # Limite operacional fixo
        'pct_acima_limite':         pct_acima_limite,
        'horas_acima_limite':       horas_acima_limite,
        # Amostragem
        'n_amostras':               n_amostras,
        'dt_medio':                 dt_medio,
        'dt_min':                   dt_min,
        'dt_variancia':             dt_variancia,
        'freq_amostras':            freq_amostras,
        # Contexto histórico
        'zscore_max':               zscore_max,
        'zscore_media':             zscore_media,
        'ratio_media':              ratio_media,
        'ratio_max':                ratio_max,
        'n_acima_threshold':        n_acima_threshold,
        'pct_acima_threshold':      pct_acima_threshold,
        'max_run_acima':            max_run_acima,
    }


# =============================================================================
# EXTRAÇÃO DE FEATURES POR JANELA
# =============================================================================

features_list      = []
valid_events       = []
eventos_timestamps = df_eventos["TIMESTAMP"]

for evento in eventos_timestamps:
    inicio_janela   = evento - pd.Timedelta(days=DIAS_JANELA)
    inicio_baseline = evento - pd.Timedelta(days=DIAS_JANELA + DIAS_BASELINE)

    # Janela principal
    mask_window = (
        (df_dataset['TIMESTAMP'] >= inicio_janela) &
        (df_dataset['TIMESTAMP'] <  evento)
    )
    df_window = df_dataset[mask_window]

    # Janela de baseline histórico
    mask_baseline = (
        (df_dataset['TIMESTAMP'] >= inicio_baseline) &
        (df_dataset['TIMESTAMP'] <  inicio_janela)
    )
    df_base = df_dataset[mask_baseline]

    baseline_arg = df_base if len(df_base) >= 2 else None

    feats = extract_features(df_window, df_baseline=baseline_arg)
    if feats:
        features_list.append(feats)
        valid_events.append(evento)

# =============================================================================
# CRIAÇÃO DO DATAFRAME DE FEATURES
# =============================================================================

df_features = pd.DataFrame(features_list, index=valid_events)

# Adiciona coluna 'Real' do df_eventos
df_real = df_eventos.set_index("TIMESTAMP")["Real"]
df_features["Real"] = df_real.reindex(df_features.index)

# Preenche NaN das features de baseline com a mediana da coluna
cols_baseline = [
    'zscore_max', 'zscore_media', 'ratio_media', 'ratio_max',
    'n_acima_threshold', 'pct_acima_threshold', 'max_run_acima'
]
for col in cols_baseline:
    if df_features[col].isna().any():
        df_features[col] = df_features[col].fillna(df_features[col].median())

# # =============================================================================
# # ISOLATION FOREST SCORE COMO FEATURE EXTRA
# # =============================================================================

# feature_cols = [col for col in df_features.columns if col != "Real"]
# X = df_features[feature_cols].values
# y = df_features["Real"].values

# X_negativos = X[y == 0]
# iso = IsolationForest(contamination=0.05, random_state=42)
# iso.fit(X_negativos)

# df_features['iso_score'] = iso.decision_function(X)

df_features

### Escalonamento dos dados

In [ ]:
# Padronização e Clusterização
scaler = StandardScaler() #StandardScaler || RobustScaler || MinMaxScaler
X_scaled = scaler.fit_transform(df_features.drop('Real', axis=1))
y = df_features['Real']

### Verificando separabilidade das amostras com PCA

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

cores = {0: 'steelblue', 1: 'crimson'}
labels_texto = {0: 'Sem Evento apontado', 1: 'Evento apontado'}

fig, ax = plt.subplots(figsize=(8, 6))
for val in [0, 1]:
    mask = df_features['Real'] == val
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=cores[val], label=labels_texto[val],
               s=100, edgecolors='k', linewidths=0.8)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variância)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variância)')
ax.legend()
ax.set_title('Separabilidade das classes')
plt.tight_layout()
plt.show()

## KMeans

In [ ]:
# Elbow Method
inercia = []
K_range = range(1, 8)

for k in K_range:
    kmeans_teste = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_teste.fit(X_scaled)
    inercia.append(kmeans_teste.inertia_)

# Plota o gráfico para visualização
plt.figure(figsize=(10,5))
plt.plot(K_range, inercia, marker='o', linestyle='--')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia')
plt.title('Método do Cotovelo para K Ideal')
plt.xticks(K_range)
plt.grid(True)
plt.show()

In [ ]:
# Clusterização
kmeans = KMeans(n_clusters=2, random_state=42, n_init='auto') # Como são duas classes, o número de clusters foi fixado como 2
df_features['Cluster'] = kmeans.fit_predict(X_scaled)
# Resultado final
df_features

In [ ]:
# Dicionário de cores dos clusters
cores_clusters = {
    0: 'blue',   
    1: 'green',
    2: 'red'
}

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_dataset['TIMESTAMP'],
    y=df_dataset["Resultado de Ferro (ppm)"],
    mode='lines',
    name="Resultado de Ferro (ppm)",
    line=dict(color='black')
))
fig.add_hline(y=5, line_width=3, line_dash="dash", line_color="red")

for _, row in df_eventos.iterrows():
    evento_ts = row["TIMESTAMP"]
    
    # Adiciona o sombreado da janela baseado no Cluster
    if evento_ts in df_features.index:
        cluster = df_features.loc[evento_ts, 'Cluster']
        cor_fundo = cores_clusters.get(cluster, 'gray')
        inicio_janela = evento_ts - pd.Timedelta(days=DIAS_JANELA)
        
        fig.add_vrect(
            x0=inicio_janela,
            x1=evento_ts,
            fillcolor=cor_fundo,
            opacity=0.2, 
            layer="below", 
            line_width=0,
            annotation_text=f"C{cluster}",
            annotation_position="top left"
        )

    # Linha vertical exata do evento
    cor = "red" if row["Real"] == 1 else "blue"
    fig.add_shape(
        type="line",
        x0=str(row["TIMESTAMP"]),
        x1=str(row["TIMESTAMP"]),
        y0=0,
        y1=1,
        yref="paper",
        line=dict(color=cor, width=1.5, dash="dash")
    )

    # Anotação
    fig.add_annotation(
        x=str(evento_ts),
        y=1,
        yref="paper",
        text=row["EVENTO"] if "EVENTO" in df_eventos.columns else "",
        showarrow=False,
        textangle=-90,
        yanchor="top"
    )

fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    title='Teor de Fe - Clusters'
)
fig.show()

In [ ]:
# Tabela de contingência
pd.crosstab(df_features['Cluster'], df_features['Real'], 
            rownames=['Cluster'], colnames=['Real'])

In [ ]:
## Quão bem os clusters recuperam os labels reais
ari  = adjusted_rand_score(df_features['Real'], df_features['Cluster'])
nmi  = normalized_mutual_info_score(df_features['Real'], df_features['Cluster'])

print(f"Adjusted Rand Score: {ari}")
print(f"Normalized Mutual Info Score: {ari}")

## ROCKET

In [ ]:
N_PONTOS_INTERP  = 50     # pontos da grade uniforme após interpolação
                          # regra prática: >= mediana de amostras por janela
N_KERNELS_ROCKET = 10000 # número de kernels aleatórios do ROCKET
N_CLUSTERS       = 2      # para o KMeans pós-ROCKET

In [ ]:
def interpolar_janela(df_window, n_pontos=N_PONTOS_INTERP):
    """
    Interpola uma janela de comprimento variável para n_pontos uniformes.

    Retorna array 1-D de shape (n_pontos,) ou None se dados insuficientes.
    """
    if len(df_window) < 2:
        return None

    t = (df_window["TIMESTAMP"] - df_window["TIMESTAMP"].min()) \
            .dt.total_seconds().values.astype(float)
    y = df_window["Resultado de Ferro (ppm)"].values.astype(float)

    # Remove duplicatas de timestamp (mantém último valor)
    _, idx_uniq = np.unique(t, return_index=True)
    t, y = t[idx_uniq], y[idx_uniq]

    if len(t) < 2:
        return None

    t_uniforme = np.linspace(t[0], t[-1], n_pontos)
    y_interp   = np.interp(t_uniforme, t, y)   # interpolação linear

    return y_interp.astype(np.float32)


# =============================================================================
# EXTRAÇÃO DAS JANELAS E INTERPOLAÇÃO
# =============================================================================

series_list  = []   # cada elemento: array 1-D de shape (N_PONTOS_INTERP,)
valid_events = []
real_labels  = []

for _, ev_row in df_eventos.iterrows():
    evento        = ev_row["TIMESTAMP"]
    inicio_janela = evento - pd.Timedelta(days=DIAS_JANELA)

    mask = (
        (df_dataset["TIMESTAMP"] >= inicio_janela) &
        (df_dataset["TIMESTAMP"] <  evento)
    )
    df_window = df_dataset[mask]

    serie = interpolar_janela(df_window)
    if serie is not None:
        series_list.append(serie)
        valid_events.append(evento)
        real_labels.append(ev_row["Real"])

print(f"Janelas válidas para o ROCKET: {len(series_list)}")

In [ ]:
X_series = np.stack(series_list)                    # (n, N_PONTOS_INTERP)
X_3d = X_series[:, np.newaxis, :]               # (n, 1, N_PONTOS_INTERP)
y = np.array(real_labels)

print(f"Shape entrada ROCKET: {X_3d.shape}")

In [ ]:
rocket = Rocket()
rocket.fit(X_3d)
X_rocket = rocket.transform(X_3d)   # (n, 2 * N_KERNELS_ROCKET)

print(f"Shape features ROCKET: {X_rocket.shape}  "
      f"({X_rocket.shape[1]} features para {X_rocket.shape[0]} amostras)")

# Normalização — obrigatória antes do Ridge e recomendada para clustering
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_rocket)

In [ ]:
print("\n" + "=" * 55)
print("CLUSTERING (KMeans sobre features ROCKET)")
print("=" * 55)

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

df_result = pd.DataFrame({
    "TIMESTAMP": valid_events,
    "Real":      y,
    "Cluster":   clusters
})

print("\nTabela de contingência:")
print(pd.crosstab(df_result["Cluster"], df_result["Real"],
                  rownames=["Cluster"], colnames=["Real"],
                  margins=True, margins_name="Total"))

ari = adjusted_rand_score(y, clusters)
nmi = normalized_mutual_info_score(y, clusters)
print(f"\nAdjusted Rand Score : {ari:.4f}")
print(f"Normalized MI Score : {nmi:.4f}")

In [ ]:
# =============================================================================
# ABORDAGEM 2 — CLASSIFICAÇÃO SUPERVISIONADA (Ridge sobre features ROCKET)
# =============================================================================
# RidgeClassifierCV é o classificador canônico recomendado com ROCKET:
# rápido, regularizado e funciona bem com features de alta dimensionalidade.

print("\n" + "=" * 55)
print("CLASSIFICAÇÃO (RidgeClassifierCV sobre features ROCKET)")
print("=" * 55)

ridge = RidgeClassifierCV(
    alphas=np.logspace(-4, 4, 20),
    class_weight="balanced"   # compensa desbalanceamento 1:9
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_validate(
    ridge, X_scaled, y, cv=cv,
    scoring={
        "f1":      make_scorer(f1_score, zero_division=0),
        "roc_auc": "roc_auc",
        "pr_auc":  "average_precision",
    }
)

print(f"\n  F1-score : {scores['test_f1'].mean():.3f} ± {scores['test_f1'].std():.3f}")
print(f"  ROC-AUC  : {scores['test_roc_auc'].mean():.3f} ± {scores['test_roc_auc'].std():.3f}")
print(f"  PR-AUC   : {scores['test_pr_auc'].mean():.3f} ± {scores['test_pr_auc'].std():.3f}")

# Treino final para o relatório completo
ridge.fit(X_scaled, y)
y_pred = ridge.predict(X_scaled)
print("\nRelatório de classificação (treino completo):")
print(classification_report(y, y_pred,
                             target_names=["Falso Positivo", "Contaminação Real"],
                             zero_division=0))

# =============================================================================
# VISUALIZAÇÃO — PCA 2D das features ROCKET colorido por Real e Cluster
# =============================================================================

pca    = PCA(n_components=2, random_state=42)
X_pca  = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("ROCKET — Espaço de Features (PCA 2D)", fontsize=13, fontweight="bold")

# Plot por label real
cores_real  = {0: "steelblue", 1: "crimson"}
nomes_real  = {0: "Falso Positivo", 1: "Contaminação Real"}
for val in [0, 1]:
    mask = y == val
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=cores_real[val], label=nomes_real[val],
                    s=80, edgecolors="k", linewidths=0.5, alpha=0.85)
axes[0].set_title("Colorido por label real")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variância)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variância)")
axes[0].legend(); axes[0].grid(alpha=0.3)

# Plot por cluster
for cl in np.unique(clusters):
    mask = clusters == cl
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    label=f"Cluster {cl}",
                    s=80, edgecolors="k", linewidths=0.5, alpha=0.85)
axes[1].set_title("Colorido por cluster KMeans")
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variância)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variância)")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## kShape

In [ ]:
N_PONTOS_INTERP = 50    # mesmo critério do ROCKET: >= mediana de amostras/janela
N_CLUSTERS      = 2

In [ ]:
# =============================================================================
# INTERPOLAÇÃO — mesma lógica do ROCKET
# =============================================================================

def interpolar_janela(df_window, n_pontos=N_PONTOS_INTERP):
    if len(df_window) < 2:
        return None
    t = (df_window["TIMESTAMP"] - df_window["TIMESTAMP"].min()) \
            .dt.total_seconds().values.astype(float)
    y = df_window["Resultado de Ferro (ppm)"].values.astype(float)
    _, idx_uniq = np.unique(t, return_index=True)
    t, y = t[idx_uniq], y[idx_uniq]
    if len(t) < 2:
        return None
    t_uniforme = np.linspace(t[0], t[-1], n_pontos)
    return np.interp(t_uniforme, t, y).astype(np.float64)

# =============================================================================
# EXTRAÇÃO DAS JANELAS
# =============================================================================

series_list  = []
valid_events = []
real_labels  = []

for _, ev_row in df_eventos.iterrows():
    evento        = ev_row["TIMESTAMP"]
    inicio_janela = evento - pd.Timedelta(days=DIAS_JANELA)
    mask = (
        (df_dataset["TIMESTAMP"] >= inicio_janela) &
        (df_dataset["TIMESTAMP"] <  evento)
    )
    serie = interpolar_janela(df_dataset[mask])
    if serie is not None:
        series_list.append(serie)
        valid_events.append(evento)
        real_labels.append(ev_row["Real"])

print(f"Janelas válidas: {len(series_list)}")

In [ ]:
# =============================================================================
# FORMATAÇÃO PARA O kShape
# =============================================================================
# tslearn espera shape (n_amostras, n_pontos, 1) — note a diferença do ROCKET
# que esperava (n_amostras, 1, n_pontos). Aqui o canal fica na última dimensão.

X_series = np.stack(series_list)           # (n, N_PONTOS_INTERP)
X_3d     = X_series[:, :, np.newaxis]      # (n, N_PONTOS_INTERP, 1)
y        = np.array(real_labels)

# Normalização z-score por série — OBRIGATÓRIA para o kShape
# O SBD é invariante a escala, mas o kShape converge melhor com séries normalizadas
scaler   = TimeSeriesScalerMeanVariance()
X_scaled = scaler.fit_transform(X_3d)      # (n, N_PONTOS_INTERP, 1)

print(f"Shape entrada kShape: {X_scaled.shape}")

In [ ]:
# =============================================================================
# CLUSTERING kShape
# =============================================================================

ks = KShape(
    n_clusters=N_CLUSTERS,
    n_init=10,              # múltiplas inicializações — kShape é sensível a isso
    max_iter=100,
    random_state=42,
    verbose=True
)
clusters = ks.fit_predict(X_scaled)

In [ ]:
# =============================================================================
# VALIDAÇÃO EXTERNA
# =============================================================================

print("\n" + "=" * 50)
print("VALIDAÇÃO DO CLUSTERING")
print("=" * 50)

print("\nTabela de contingência:")
df_result = pd.DataFrame({
    "TIMESTAMP": valid_events,
    "Real":      y,
    "Cluster":   clusters
})
print(pd.crosstab(
    df_result["Cluster"], df_result["Real"],
    rownames=["Cluster"], colnames=["Real"],
    margins=True, margins_name="Total"
))

ari = adjusted_rand_score(y, clusters)
nmi = normalized_mutual_info_score(y, clusters)
print(f"\nAdjusted Rand Score : {ari:.4f}")
print(f"Normalized MI Score : {nmi:.4f}")

In [ ]:
fig = plt.figure(figsize=(18, 12))

# Centroides
# Os centroides são séries temporais interpretáveis: mostram o "formato típico"
# de cada cluster ao longo dos 5 dias antes do evento.
ax_cent = fig.add_subplot(2, 2, (1, 2))
t_eixo  = np.linspace(-DIAS_JANELA, 0, N_PONTOS_INTERP)  # dias antes do evento

for k in range(N_CLUSTERS):
    centroide = ks.cluster_centers_[k].ravel()
    n_real1   = ((clusters == k) & (y == 1)).sum()
    n_total   = (clusters == k).sum()
    ax_cent.plot(t_eixo, centroide, lw=2.5,
                 label=f"Cluster {k}  (n={n_total}, Real=1: {n_real1})")

ax_cent.axvline(0, color="black", linestyle="--", lw=1, alpha=0.5,
                label="Evento (t=0)")
ax_cent.set_xlabel("Dias antes do evento")
ax_cent.set_ylabel("Fe ppm (z-score normalizado)")
ax_cent.set_title("Centroides dos Clusters — Formato Típico do Sinal\n",
                  fontweight="bold")
ax_cent.legend(); ax_cent.grid(alpha=0.3)

# Séries individuais por cluster
for k in range(min(N_CLUSTERS, 2)):
    ax = fig.add_subplot(2, 2, 3 + k)
    mask_k = clusters == k
    series_k = X_scaled[mask_k, :, 0]
    labels_k = y[mask_k]

    for i, (serie, label) in enumerate(zip(series_k, labels_k)):
        cor   = "crimson"   if label == 1 else "steelblue"
        alpha = 0.8         if label == 1 else 0.3
        lw    = 2.0         if label == 1 else 0.8
        ax.plot(t_eixo, serie, color=cor, alpha=alpha, lw=lw)

    # Centroide sobre as séries
    ax.plot(t_eixo, ks.cluster_centers_[k].ravel(),
            color="black", lw=2.5, linestyle="--", label="Centroide")

    n_r1 = (labels_k == 1).sum()
    n_r0 = (labels_k == 0).sum()
    ax.set_title(f"Cluster {k}  —  "
                 f"Vermelho=Real=1 (n={n_r1})  |  Azul=Real=0 (n={n_r0})",
                 fontweight="bold")
    ax.set_xlabel("Dias antes do evento")
    ax.set_ylabel("Fe ppm (z-score)")
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle("kShape — Clustering de Séries Temporais de Ferro (ppm)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Classificação (Abordagem Supervisionada)

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
import numpy as np

X = df_features.drop(columns=['real'])
y = df_features['real']

# scale_pos_weight = n_negativos / n_positivos
modelo = XGBClassifier(
    scale_pos_weight=83/9,  # corrige o desbalanceamento
    n_estimators=100,
    max_depth=3,            # raso — evita overfitting com 92 amostras
    learning_rate=0.05,
    subsample=0.8,
    eval_metric='aucpr',    # AUC da curva Precision-Recall
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Stratified garante que cada fold tenha proporção de positivos

scores = cross_validate(modelo, X, y, cv=cv, scoring={
    'f1':      make_scorer(f1_score),
    'roc_auc': 'roc_auc',
    'pr_auc':  'average_precision'
})

print(f"F1:     {scores['test_f1'].mean():.3f} ± {scores['test_f1'].std():.3f}")
print(f"ROCAUC: {scores['test_roc_auc'].mean():.3f} ± {scores['test_roc_auc'].std():.3f}")
print(f"PR-AUC: {scores['test_pr_auc'].mean():.3f} ± {scores['test_pr_auc'].std():.3f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler

pipeline = ImbPipeline([
    ('scaler',  StandardScaler()),
    ('smote',   SMOTE(k_neighbors=3, random_state=42)),
    # k_neighbors=3 porque só temos 9 positivos
    ('modelo',  LogisticRegression(class_weight='balanced', max_iter=1000))
])

scores = cross_validate(pipeline, X, y, cv=cv, scoring={
    'f1':     make_scorer(f1_score),
    'pr_auc': 'average_precision'
})

In [ ]:
from sklearn.metrics import precision_recall_curve

modelo.fit(X, y)
probs = modelo.predict_proba(X)[:, 1]

precision, recall, thresholds = precision_recall_curve(y, probs)

# Threshold que maximiza F1
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"Threshold ótimo: {best_threshold:.3f}")
```

---

## Métricas para Usar e Evitar

| Métrica | Usar? | Motivo |
|---|---|---|
| Acurácia | ❌ | Enganosa com desbalanceamento |
| ROC-AUC | ⚠️ Cuidado | Otimista com classes desbalanceadas |
| **PR-AUC** | ✅ Principal | Honesta com desbalanceamento |
| **F1-score** | ✅ Principal | Balanceia precisão e recall |
| **Recall dos positivos** | ✅ Crítico | Falso negativo (perder contaminação real) é o pior erro |
| Precision dos positivos | ✅ Secundário | Falso alarme tem custo menor que parar desnecessariamente |

---

## Arquitetura Final Sugerida
```
92 janelas de 7 dias (features extraídas)
          │
          ├── Clustering (sem labels)
          │     └─ Validar com ARI/NMI → as features fazem sentido?
          │
          ├── Isolation Forest nos 83 negativos
          │     └─ iso_score como feature extra
          │
          ├── XGBoost com scale_pos_weight=9.2
          │     └─ StratifiedKFold(5) + PR-AUC + F1
          │     └─ Threshold tuning pela curva PR
          │
          └── SHAP values para interpretabilidade
                └─ Quais features mais separam Real=1 de Real=0?
                └─ Apresentar para equipe de manutenção

## LDA/QDA

In [ ]:
"""
Análise Discriminante Linear (LDA) e Quadrática (QDA)
para detecção de contaminação de ferro em ppm.

Contexto: 9 eventos reais (Real=1) vs 83 falsos positivos (Real=0)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_validate, LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay, make_scorer, f1_score
)
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

# =============================================================================
# CONFIGURAÇÕES
# =============================================================================

RANDOM_STATE  = 42
N_SPLITS_CV   = 5          # folds para StratifiedKFold
REG_PARAM_QDA = 1e-2       # regularização QDA (0 = sem reg, aumentar se singular)

X = df_features.drop('Real', axis=1)
y = df_features['Real']
feature_names = df_features.drop(columns=['Real']).columns.tolist()

print("=" * 60)
print("DISTRIBUIÇÃO DAS CLASSES")
print("=" * 60)
print(f"  Real=0 (falso positivo): {(y==0).sum()} amostras")
print(f"  Real=1 (contaminação):   {(y==1).sum()} amostras")
print(f"  Razão desbalanceamento:  1:{(y==0).sum()//(y==1).sum()}")
print()

# =============================================================================
# 2. DEFINIÇÃO DOS MODELOS
# =============================================================================

modelos = {
    'LDA': Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'))
        # shrinkage='auto' usa Ledoit-Wolf — robusto com poucas amostras
    ]),
    'QDA': Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    QuadraticDiscriminantAnalysis(reg_param=REG_PARAM_QDA))
        # reg_param regulariza a matriz de covariância — necessário com n pequeno
    ])
}

# =============================================================================
# 3. VALIDAÇÃO CRUZADA
# =============================================================================

print("=" * 60)
print("VALIDAÇÃO CRUZADA — StratifiedKFold (5 folds)")
print("=" * 60)

cv_strat = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)
cv_loo   = LeaveOneOut()

scoring = {
    'f1':       make_scorer(f1_score, zero_division=0),
    'roc_auc':  'roc_auc',
    'pr_auc':   'average_precision',
    'recall':   make_scorer(__import__('sklearn.metrics', fromlist=['recall_score'])
                            .recall_score, zero_division=0),
    'precision': make_scorer(__import__('sklearn.metrics', fromlist=['precision_score'])
                             .precision_score, zero_division=0),
}

resultados_cv = {}

for nome, pipeline in modelos.items():
    scores = cross_validate(pipeline, X, y, cv=cv_strat, scoring=scoring)
    resultados_cv[nome] = scores

    print(f"\n  {nome}")
    print(f"    F1-score  : {scores['test_f1'].mean():.3f} ± {scores['test_f1'].std():.3f}")
    print(f"    ROC-AUC   : {scores['test_roc_auc'].mean():.3f} ± {scores['test_roc_auc'].std():.3f}")
    print(f"    PR-AUC    : {scores['test_pr_auc'].mean():.3f} ± {scores['test_pr_auc'].std():.3f}")
    print(f"    Recall    : {scores['test_recall'].mean():.3f} ± {scores['test_recall'].std():.3f}")
    print(f"    Precision : {scores['test_precision'].mean():.3f} ± {scores['test_precision'].std():.3f}")

# =============================================================================
# 4. LEAVE-ONE-OUT — validação complementar (n pequeno)
# =============================================================================

print("\n" + "=" * 60)
print("LEAVE-ONE-OUT (LOO) — validação complementar")
print("=" * 60)

for nome, pipeline in modelos.items():
    scores_loo = cross_validate(pipeline, X, y, cv=cv_loo,
                                scoring={'f1': make_scorer(f1_score, zero_division=0),
                                         'roc_auc': 'roc_auc'})
    print(f"\n  {nome} (LOO)")
    print(f"    F1-score : {scores_loo['test_f1'].mean():.3f}")
    print(f"    ROC-AUC  : {scores_loo['test_roc_auc'].mean():.3f}")

# =============================================================================
# 5. AJUSTE FINAL E THRESHOLD TUNING
# =============================================================================

print("\n" + "=" * 60)
print("THRESHOLD TUNING — otimização pelo F1 na curva PR")
print("=" * 60)

modelos_fit   = {}
thresholds_ot = {}

for nome, pipeline in modelos.items():
    pipeline.fit(X, y)
    modelos_fit[nome] = pipeline

    probs = pipeline.predict_proba(X)[:, 1]
    precision_arr, recall_arr, thresh_arr = precision_recall_curve(y, probs)

    f1_arr  = np.where(
        (precision_arr + recall_arr) > 0,
        2 * precision_arr * recall_arr / (precision_arr + recall_arr),
        0
    )
    best_idx     = np.argmax(f1_arr)
    best_thresh  = thresh_arr[best_idx] if best_idx < len(thresh_arr) else 0.5
    thresholds_ot[nome] = best_thresh

    print(f"\n  {nome}")
    print(f"    Threshold ótimo : {best_thresh:.3f}")
    print(f"    F1 no threshold : {f1_arr[best_idx]:.3f}")

# =============================================================================
# 6. RELATÓRIO DE CLASSIFICAÇÃO COM THRESHOLD OTIMIZADO
# =============================================================================

print("\n" + "=" * 60)
print("RELATÓRIO DE CLASSIFICAÇÃO (threshold otimizado, treino completo)")
print("=" * 60)

for nome, pipeline in modelos_fit.items():
    probs  = pipeline.predict_proba(X)[:, 1]
    y_pred = (probs >= thresholds_ot[nome]).astype(int)

    print(f"\n  {nome} — threshold={thresholds_ot[nome]:.3f}")
    print(classification_report(y, y_pred,
                                 target_names=['Falso Positivo', 'Contaminação Real'],
                                 zero_division=0))

# =============================================================================
# 7. IMPORTÂNCIA DE FEATURES (permutation importance)
# =============================================================================

print("=" * 60)
print("IMPORTÂNCIA DE FEATURES — permutation importance")
print("=" * 60)

importancias = {}
for nome, pipeline in modelos_fit.items():
    perm = permutation_importance(
        pipeline, X, y,
        n_repeats=30,
        random_state=RANDOM_STATE,
        scoring='average_precision'
    )
    importancias[nome] = perm
    idx_sorted = np.argsort(perm.importances_mean)[::-1]
    print(f"\n  {nome} — top 5 features:")
    for i in idx_sorted[:5]:
        print(f"    {feature_names[i]:<25} {perm.importances_mean[i]:.4f} ± {perm.importances_std[i]:.4f}")

# =============================================================================
# 8. VISUALIZAÇÕES
# =============================================================================

fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

cores_classe = {0: '#4878CF', 1: '#D65F5F'}
nomes_classe = {0: 'Falso Positivo', 1: 'Contaminação Real'}

# ── 8.1  Matrizes de Confusão ──────────────────────────────────────────────
for col, (nome, pipeline) in enumerate(modelos_fit.items()):
    ax = fig.add_subplot(gs[0, col])
    probs  = pipeline.predict_proba(X)[:, 1]
    y_pred = (probs >= thresholds_ot[nome]).astype(int)
    cm = confusion_matrix(y, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Falso +', 'Real'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{nome}\nMatriz de Confusão (thr={thresholds_ot[nome]:.2f})',
                 fontsize=11, fontweight='bold')

# ── 8.2  Curvas ROC ────────────────────────────────────────────────────────
ax_roc = fig.add_subplot(gs[0, 2])
for nome, pipeline in modelos_fit.items():
    probs = pipeline.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y, probs)
    roc_auc_val = auc(fpr, tpr)
    ax_roc.plot(fpr, tpr, lw=2, label=f'{nome} (AUC={roc_auc_val:.3f})')
ax_roc.plot([0,1],[0,1],'k--', lw=1, label='Aleatório')
ax_roc.set_xlabel('Taxa Falso Positivo'); ax_roc.set_ylabel('Taxa Verdadeiro Positivo')
ax_roc.set_title('Curva ROC', fontweight='bold')
ax_roc.legend(fontsize=9); ax_roc.grid(alpha=0.3)

# ── 8.3  Curvas Precision-Recall ───────────────────────────────────────────
ax_pr = fig.add_subplot(gs[1, 0])
for nome, pipeline in modelos_fit.items():
    probs = pipeline.predict_proba(X)[:, 1]
    prec, rec, _ = precision_recall_curve(y, probs)
    ap = average_precision_score(y, probs)
    ax_pr.plot(rec, prec, lw=2, label=f'{nome} (AP={ap:.3f})')
ax_pr.axhline(y=(y==1).mean(), color='k', linestyle='--', lw=1, label='Baseline')
ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
ax_pr.set_title('Curva Precision-Recall\n(mais informativa com desbalanceamento)',
                fontweight='bold')
ax_pr.legend(fontsize=9); ax_pr.grid(alpha=0.3)

# ── 8.4  Scores de probabilidade por classe ────────────────────────────────
for col, (nome, pipeline) in enumerate(modelos_fit.items()):
    ax = fig.add_subplot(gs[1, col + 1])
    probs = pipeline.predict_proba(X)[:, 1]
    for cls in [0, 1]:
        mask = y == cls
        ax.hist(probs[mask], bins=15, alpha=0.6,
                color=cores_classe[cls], label=nomes_classe[cls], edgecolor='white')
    ax.axvline(thresholds_ot[nome], color='black', linestyle='--', lw=1.5,
               label=f'Threshold={thresholds_ot[nome]:.2f}')
    ax.set_xlabel('Probabilidade predita (classe Real=1)')
    ax.set_ylabel('Contagem')
    ax.set_title(f'{nome}\nDistribuição de Scores', fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ── 8.5  Importância de Features ──────────────────────────────────────────
for col, (nome, perm) in enumerate(importancias.items()):
    ax = fig.add_subplot(gs[2, col])
    idx_sorted = np.argsort(perm.importances_mean)[::-1][:10]
    y_pos = np.arange(len(idx_sorted))
    ax.barh(y_pos, perm.importances_mean[idx_sorted], xerr=perm.importances_std[idx_sorted],
            color='steelblue', alpha=0.8, edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels([feature_names[i] for i in idx_sorted], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Redução em PR-AUC (permutação)')
    ax.set_title(f'{nome}\nTop 10 Features', fontweight='bold')
    ax.grid(alpha=0.3, axis='x')

# ── 8.6  Comparação CV (F1 por fold) ──────────────────────────────────────
ax_cv = fig.add_subplot(gs[2, 2])
for nome, scores in resultados_cv.items():
    ax_cv.plot(range(1, N_SPLITS_CV + 1), scores['test_f1'],
               marker='o', lw=2, label=nome)
ax_cv.set_xlabel('Fold'); ax_cv.set_ylabel('F1-score')
ax_cv.set_title('F1-score por Fold (StratifiedKFold)', fontweight='bold')
ax_cv.set_xticks(range(1, N_SPLITS_CV + 1))
ax_cv.legend(); ax_cv.grid(alpha=0.3)

fig.suptitle('Análise Discriminante: LDA vs QDA\nDetecção de Contaminação de Ferro',
             fontsize=14, fontweight='bold', y=1.01)

plt.show()

# =============================================================================
# 9. RESUMO FINAL
# =============================================================================

print("\n" + "=" * 60)
print("RESUMO COMPARATIVO")
print("=" * 60)
print(f"{'Modelo':<8} {'F1':>8} {'ROC-AUC':>10} {'PR-AUC':>10} {'Recall':>9} {'Threshold':>11}")
print("-" * 60)
for nome, scores in resultados_cv.items():
    print(f"{nome:<8} "
          f"{scores['test_f1'].mean():>8.3f} "
          f"{scores['test_roc_auc'].mean():>10.3f} "
          f"{scores['test_pr_auc'].mean():>10.3f} "
          f"{scores['test_recall'].mean():>9.3f} "
          f"{thresholds_ot[nome]:>11.3f}")